# Risk-aware Feature-conditioned 1D-CNN CVAE for Dangerous ECG Waveform Generation

This notebook trains a **conditional VAE (CVAE)** whose explicit goal is to
generate *risky / dangerous / true-alarm* abnormal 2-channel ECG waveforms —
**not** generic ECG-like signals.

It is **not** a plain VAE: it injects **handcrafted risk descriptors** as
conditioning and is optimized with several **risk-aware losses** so that the
generated waveforms reproduce clinically dangerous structure such as
high-amplitude spikes, sharp slopes, abnormal burst segments, rapid repeated
beats, high-frequency oscillations, abnormal rhythm changes, and inter-lead
(ECG1 vs ECG2) consistency.

**Input ECG shape:** `(B, 2, 2500)` — channel 0 = ECG1, channel 1 = ECG2,
length = 2500 time steps. **Labels:** `0 = normal`, `1 = dangerous / true alarm`.

### Key components
- 2-channel ECG input
- 1D-CNN encoder / decoder (no `ConvTranspose1d` — uses `Upsample + Conv1d`)
- **Shared** latent space + **risk-specific** latent space
- Label conditioning (embedding)
- **Handcrafted risk-feature conditioning** (amplitude, slope, FFT, rhythm, inter-lead)
- **Event-weighted** reconstruction loss (focus on risky regions)
- **Derivative** loss (preserve sharp slopes)
- **Feature-consistency** loss (differentiable risk features of real vs generated)
- **Auxiliary risk-classification** loss (generated ECG keeps its label identity)
- Conditional generation of **normal** or **dangerous** ECG, with a `risk_strength` knob

### How to run
1. `Runtime > Change runtime type > GPU` (T4 is plenty).
2. Run every cell from top to bottom. With `USE_SYNTHETIC_DEBUG = True` the
   whole pipeline runs end-to-end **without any external files**.
3. To use real data, set `USE_SYNTHETIC_DEBUG = False` and provide
   `ecg.npy` `(N, 2, 2500)` and `labels.npy` `(N,)`.

## 2. Colab setup
Checks the GPU, imports core libraries, installs anything missing, sets a global
seed, and creates the output directories under `/content/risk_ecg_cvae_outputs`.

In [ ]:
import os, sys, random, subprocess, math

import numpy as np

# --- make sure third-party deps exist (Colab already has torch + matplotlib) ---
def _ensure(pkg, import_name=None):
    name = import_name or pkg
    try:
        __import__(name)
    except Exception:
        print(f"Installing {pkg} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

_ensure("torch")
_ensure("matplotlib")
_ensure("tqdm")

import torch
import matplotlib
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT_DIR = "/content/risk_ecg_cvae_outputs"

print("Torch version :", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU           :", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. Enable it via Runtime > Change runtime type > GPU.")
    print("(CPU still works for the synthetic debug run, just slower.)")

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

for sub in ["", "samples", "checkpoints", "generated"]:
    os.makedirs(os.path.join(OUTPUT_DIR, sub), exist_ok=True)
print("Output dir    :", OUTPUT_DIR)
print("Device        :", DEVICE)

### (Optional) Mount Google Drive
Uncomment if you want to read real `.npy` data from / save checkpoints to Drive.

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")


## 3. Configuration
Everything is in one `CONFIG` dict. The two values you will most likely tweak —
`epochs` and `batch_size` — are also exposed as plain variables right below for
convenience.

In [ ]:
CONFIG = {
    "data": {
        "input_channels": 2,
        "input_length": 2500,
        "train_ratio": 0.8,
        "batch_size": 32,
        "num_workers": 2,
    },
    "model": {
        "shared_latent_dim": 64,
        "risk_latent_dim": 64,
        "label_emb_dim": 16,
        "feature_emb_dim": 64,
        "base_channels": 32,
        "final_activation": None,   # None or "tanh"
    },
    "loss": {
        "lambda_event": 2.0,
        "lambda_derivative": 0.5,
        "lambda_feature": 0.2,
        "lambda_classifier": 0.5,
        "beta_shared_max": 0.001,
        "beta_risk_max": 0.001,
        "kl_anneal_epochs": 30,
        "event_weight": 5.0,
    },
    "train": {
        "epochs": 50,
        "learning_rate": 2e-4,
        "weight_decay": 1e-4,
        "grad_clip": 1.0,
        "seed": 42,
        "sample_every": 5,
        "save_every": 10,
        "use_amp": False,
    },
    "synthetic": {
        "enabled": True,
        "num_samples": 1000,
        "dangerous_ratio": 0.5,
    },
}

# ---- quick knobs (override the most-tweaked settings here) ----
EPOCHS     = CONFIG["train"]["epochs"]        # e.g. set to 30 for a fast run
BATCH_SIZE = CONFIG["data"]["batch_size"]     # e.g. 16 if you hit OOM
CONFIG["train"]["epochs"]    = EPOCHS
CONFIG["data"]["batch_size"] = BATCH_SIZE

set_seed(CONFIG["train"]["seed"])
print("CONFIG ready. epochs =", EPOCHS, "| batch_size =", BATCH_SIZE)


# ===== improvement knobs (stratified split / weighted sampler / finetune / modes / generate) =====
CONFIG["train"]["use_weighted_sampler"] = True      # 개선 4: 클래스 균형 샘플링
CONFIG["train"]["use_true_finetune"]    = False     # 개선 7: 2단계 true-only 미세조정
CONFIG["train"]["true_finetune_epochs"] = 10
CONFIG["train"]["true_finetune_lr"]     = 1e-4
CONFIG["loss"].setdefault("lambda_classifier", 0.5) # 개선 6: 라벨 보존 분류 손실 가중치
# 개선 8: 실험 모드 (conditional_all | true_only | conditional_all_then_true_finetune)
CONFIG["experiment"] = {"mode": "conditional_all"}
# 개선 9: 생성 설정
CONFIG["generate"] = {"num_samples": 256, "dangerous_risk_strength": 1.5, "normal_risk_strength": 1.0,
                      "auto_fill_deficit": True, "fill_buffer": 64}  # dangerous 는 train 불균형(deficit) 만큼 자동 생성

# mode -> finetune 자동 연결
if CONFIG["experiment"]["mode"] == "conditional_all_then_true_finetune":
    CONFIG["train"]["use_true_finetune"] = True
assert CONFIG["experiment"]["mode"] in ("conditional_all", "true_only", "conditional_all_then_true_finetune")

print("experiment mode      :", CONFIG["experiment"]["mode"])
print("weighted sampler     :", CONFIG["train"]["use_weighted_sampler"])
print("true-alarm finetune  :", CONFIG["train"]["use_true_finetune"],
      f"(epochs={CONFIG['train']['true_finetune_epochs']}, lr={CONFIG['train']['true_finetune_lr']})")
print("generate             :", CONFIG["generate"])


## 4. Data: VTaC loader + synthetic debug generator

**현재 ECG CVAE 학습 데이터 동작 (개선 1 — 명시적 문서화)**

```python
Xv = d["X"].float().numpy()   # (N, 4, 2500) : ECG1, ECG2, PPG, ABP
yv = d["y"].long().numpy()    # 0 = false alarm, 1 = true alarm
X  = Xv[:, 0:2, :]            # ECG1, ECG2 만 사용
y  = yv                       # 라벨은 버리지 않고 조건(condition)으로 사용
```

- ECG CVAE 는 **ECG1, ECG2 두 채널만** 입력으로 사용한다.
- **false alarm 과 true alarm 샘플을 모두** 학습에 사용한다 (`y` 를 버리지 않음).
- `label = 0` → false alarm / 비-위험 파형, `label = 1` → true alarm / 위험(dangerous) 파형.
- 모델의 목표는 조건부 분포 **`p(ECG | label, risk_features)`** 학습이다.

**왜 true/false 를 같이 쓰는가 (개선 2 — 의도적 설계)**

기본 학습은 두 클래스를 모두 쓰는 **조건부 학습**(`conditional_all`)이며, 이는 의도된 것이다:

- true alarm 샘플은 **희소**하다 → true-only 만 쓰면 **과적합·다양성 저하** 위험.
- 두 클래스를 모두 쓰면 CVAE 가 **ECG 전체 분포**를 학습하고, 디코더가 라벨 조건에 따라
  false-like / true-like 파형을 모두 생성할 수 있다.
- **최종 목표 생성은 여전히 `label = 1`** 로 수행한다 (위험 파형 증강).
- true-only / true-finetune 비교는 `CONFIG["experiment"]["mode"]` 로 선택 가능 (개선 7·8).

합성 모드(`DATA_SOURCE="synthetic"`)는 디버그용으로 그대로 유지된다.

In [ ]:
# ----------------------- data source switch -----------------------
# "vtac"      : 실제 VTaC 4채널 .pt 에서 ECG1/ECG2 추출 (권장 - classifier 와 동일 데이터)
# "synthetic" : 합성 디버그 데이터 (드라이브/파일 불필요)
DATA_SOURCE = "vtac"
VTAC_TRAIN_PT   = "/content/drive/MyDrive/vtac_project/data/processed/train_10s.pt"
USE_SYNTHETIC_DEBUG = (DATA_SOURCE == "synthetic")   # 하위 호환
# ------------------------------------------------------------------

def _add_beat(sig, center, amp, width, rng):
    """Add a single P-QRS-T-like beat centered at `center` into `sig` (in place)."""
    L = len(sig)
    span = int(60 * width)
    lo, hi = max(0, center - span), min(L, center + span)
    if hi <= lo:
        return sig
    t = np.arange(lo, hi) - center
    p  =  0.10 * np.exp(-0.5 * ((t + 32 * width) / (8.0 * width)) ** 2)
    q  = -0.15 * np.exp(-0.5 * ((t +  6 * width) / (3.0 * width)) ** 2)
    r  =  1.00 * np.exp(-0.5 * ((t            ) / (3.0 * width)) ** 2)
    s  = -0.25 * np.exp(-0.5 * ((t -  6 * width) / (3.0 * width)) ** 2)
    tw =  0.22 * np.exp(-0.5 * ((t - 42 * width) / (12.0 * width)) ** 2)
    sig[lo:hi] += amp * (p + q + r + s + tw).astype(np.float32)
    return sig

def _base_rhythm(length, rng, rr=300, rr_jitter=12, amp=1.0, width=1.0):
    """A regular beat train with mild baseline wander + noise."""
    sig = np.zeros(length, dtype=np.float32)
    phase = rng.uniform(0, 2 * np.pi)
    sig += (0.05 * amp) * np.sin(2 * np.pi * np.arange(length) / (length / 2.0) + phase)
    pos = int(rng.integers(0, rr))
    while pos < length:
        _add_beat(sig, pos, amp * rng.uniform(0.9, 1.1), width, rng)
        pos += int(max(40, rr + rng.normal(0, rr_jitter)))
    sig += rng.normal(0, 0.02, size=length).astype(np.float32)
    return sig

def _second_lead(ch1, rng):
    """ECG2: correlated with ECG1 (scaled + small lag) but not identical."""
    lag = int(rng.integers(1, 5))
    a = rng.uniform(0.6, 0.95)
    ch2 = a * np.roll(ch1, lag)
    ch2 += rng.normal(0, 0.03, size=ch1.shape).astype(np.float32)
    ch2 += (0.04) * np.sin(2 * np.pi * np.arange(len(ch1)) / (len(ch1) / 3.0)
                           + rng.uniform(0, 2 * np.pi))
    return ch2.astype(np.float32)

def _inject_danger(ch, rng):
    """Add risky abnormal events to a single channel (in place-ish, returns new)."""
    L = len(ch)
    ch = ch.copy()
    n_events = int(rng.integers(2, 5))
    for _ in range(n_events):
        kind = rng.integers(0, 4)
        if kind == 0:
            # high-amplitude spike burst
            c = int(rng.integers(0, L))
            for k in range(int(rng.integers(3, 8))):
                pos = c + int(rng.normal(0, 25))
                if 0 <= pos < L:
                    w = rng.uniform(1.0, 3.0)
                    a = rng.uniform(2.0, 4.5) * rng.choice([-1.0, 1.0])
                    lo, hi = max(0, pos - 8), min(L, pos + 8)
                    tt = np.arange(lo, hi) - pos
                    ch[lo:hi] += (a * np.exp(-0.5 * (tt / w) ** 2)).astype(np.float32)
        elif kind == 1:
            # rapid repeated abnormal beats (VT-like run)
            start = int(rng.integers(0, max(1, L - 600)))
            rr = int(rng.integers(55, 110))
            pos = start
            end = min(L, start + int(rng.integers(400, 800)))
            while pos < end:
                _add_beat(ch, pos, rng.uniform(1.8, 3.0), rng.uniform(0.6, 0.9), rng)
                pos += rr
        elif kind == 2:
            # large slow deflection
            start = int(rng.integers(0, max(1, L - 300)))
            seg = int(rng.integers(120, 300))
            tt = np.linspace(-np.pi, np.pi, seg)
            ch[start:start + seg] += (rng.uniform(2.0, 3.5)
                                      * np.sign(rng.normal())
                                      * np.sin(tt)).astype(np.float32)[:max(0, min(seg, L - start))]
        else:
            # high-frequency oscillation burst
            start = int(rng.integers(0, max(1, L - 250)))
            seg = int(rng.integers(100, 250))
            tt = np.arange(seg)
            freq = rng.uniform(0.25, 0.45)
            ch[start:start + seg] += (rng.uniform(0.8, 1.6)
                                      * np.sin(2 * np.pi * freq * tt)).astype(np.float32)[:max(0, min(seg, L - start))]
    return ch

def generate_synthetic_ecg_dataset(num_samples, length=2500, dangerous_ratio=0.5, seed=0):
    """Return X (N,2,length) float32 and y (N,) int64 with 0=normal, 1=dangerous."""
    rng = np.random.default_rng(seed)
    X = np.zeros((num_samples, 2, length), dtype=np.float32)
    y = np.zeros((num_samples,), dtype=np.int64)
    n_danger = int(round(num_samples * dangerous_ratio))
    danger_idx = set(rng.choice(num_samples, size=n_danger, replace=False).tolist())
    for i in range(num_samples):
        if i in danger_idx:
            rr = int(rng.integers(240, 340))
            ch1 = _base_rhythm(length, rng, rr=rr, rr_jitter=14, amp=0.7, width=1.0)
            ch1 = _inject_danger(ch1, rng)
            ch2 = _second_lead(ch1, rng)
            ch2 = _inject_danger(ch2, rng) if rng.random() < 0.5 else _second_lead(ch1, rng)
            X[i, 0], X[i, 1], y[i] = ch1, ch2, 1
        else:
            rr = int(rng.integers(260, 360))
            ch1 = _base_rhythm(length, rng, rr=rr, rr_jitter=8, amp=0.6, width=1.0)
            ch2 = _second_lead(ch1, rng)
            X[i, 0], X[i, 1], y[i] = ch1, ch2, 0
    return X, y


def load_vtac_ecg_data(pt_path):
    """VTaC 4채널 .pt -> (N,2,2500) ECG1/ECG2 + (N,) 라벨(0=false alarm, 1=true alarm=dangerous)."""
    if not os.path.exists(pt_path):
        raise FileNotFoundError(
            f"VTaC 파일 없음: '{pt_path}'. 드라이브 마운트/경로 확인, 또는 DATA_SOURCE='synthetic'.")
    try:
        d = torch.load(pt_path, map_location="cpu", weights_only=False)
    except TypeError:
        d = torch.load(pt_path, map_location="cpu")
    Xv = d["X"].float().numpy()                       # (N,4,2500): ECG1,ECG2,PPG,ABP
    yv = d["y"].long().numpy().reshape(-1)            # 0=false alarm, 1=true alarm
    assert Xv.ndim == 3 and Xv.shape[1] >= 2, f"VTaC X must be (N,>=2,L), got {Xv.shape}"
    X = np.ascontiguousarray(Xv[:, 0:2, :]).astype(np.float32)   # ECG1, ECG2 만 사용
    y = yv.astype(np.int64)
    assert set(np.unique(y).tolist()).issubset({0, 1}), "labels must be 0/1"
    print(f"VTaC loaded: {pt_path}")
    print(f"  full {Xv.shape} -> ECG-only {X.shape} | true(=dangerous) {int((y==1).sum())}/{len(y)}")
    return X, y


if DATA_SOURCE == "vtac":
    try:
        from google.colab import drive as _drv
        _drv.mount("/content/drive")
    except Exception as _e:
        print("drive mount note:", _e)
    X, y = load_vtac_ecg_data(VTAC_TRAIN_PT)
    print("VTaC ECG data loaded.")
elif DATA_SOURCE == "synthetic":
    X, y = generate_synthetic_ecg_dataset(
        CONFIG["synthetic"]["num_samples"],
        length=CONFIG["data"]["input_length"],
        dangerous_ratio=CONFIG["synthetic"]["dangerous_ratio"],
        seed=CONFIG["train"]["seed"])
    print("Synthetic debug data generated.")
else:
    raise ValueError(f"DATA_SOURCE must be 'vtac' or 'synthetic', got {DATA_SOURCE!r}")

print("X shape:", X.shape, "| y shape:", y.shape)
print("normal:", int((y == 0).sum()), "| dangerous:", int((y == 1).sum()))

print("-" * 60)
print("[데이터 동작 요약]")
print("  입력 채널 : ECG1, ECG2 (X[:, 0:2, :]) ->", X.shape)
print("  라벨 사용 : 0=false alarm(normal), 1=true alarm(dangerous) | 라벨 보존됨")
print("  학습 대상 : false + true 모두 (조건부 p(ECG|label, risk_features))")
print("  최종 생성 : label=1 (true-alarm 위험 파형)")
print("-" * 60)


## 5. Visualization
`plot_ecg_sample` draws ECG1 (blue, top) and ECG2 (red, bottom). We preview a few
normal and dangerous examples before training.

In [ ]:
def plot_ecg_sample(x, label=None, title=None, save_path=None, idx=None):
    """x: (2, L) array/tensor. ECG1 on top (blue), ECG2 on bottom (red)."""
    if hasattr(x, "detach"):
        x = x.detach().cpu().numpy()
    x = np.asarray(x)
    assert x.shape[0] == 2, f"expected (2, L), got {x.shape}"
    L = x.shape[1]
    lab_txt = {0: "normal", 1: "DANGEROUS", None: "?"}.get(int(label) if label is not None else None, str(label))
    head = title or "ECG sample"
    if idx is not None:
        head += f" (idx={idx})"
    head += f"  |  label={lab_txt}"

    fig, axes = plt.subplots(2, 1, figsize=(13, 4.5), sharex=True)
    axes[0].plot(np.arange(L), x[0], color="tab:blue", lw=0.9)
    axes[0].set_ylabel("ECG1"); axes[0].grid(True, alpha=0.3)
    axes[0].set_title(head)
    axes[1].plot(np.arange(L), x[1], color="tab:red", lw=0.9)
    axes[1].set_ylabel("ECG2"); axes[1].set_xlabel("time step"); axes[1].grid(True, alpha=0.3)
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=110, bbox_inches="tight")
    plt.show()
    plt.close(fig)

# preview a couple of each class
_norm_ids = np.where(y == 0)[0][:2]
_dang_ids = np.where(y == 1)[0][:2]
for i in _norm_ids:
    plot_ecg_sample(X[i], label=y[i], title="REAL normal", idx=int(i))
for i in _dang_ids:
    plot_ecg_sample(X[i], label=y[i], title="REAL dangerous", idx=int(i))

## 6. Handcrafted risk-feature extraction (numpy + torch)
For every 2-channel window we compute a fixed-length vector of risk descriptors.
Per channel: amplitude range, max-abs amplitude, signal energy, slope energy,
max-abs slope, high-/low-frequency power (via FFT), high/low ratio, zero-crossing
rate, rough peak density. Across channels: inter-lead correlation.

- **numpy** version → dataset preprocessing & conditioning.
- **torch** version → *differentiable* feature-consistency loss during training.

Both use the **same ordering** so the two are directly comparable.

In [ ]:
FEATURE_NAMES = []
for ch in [1, 2]:
    FEATURE_NAMES += [
        f"ch{ch}_amplitude_range",
        f"ch{ch}_max_abs_amplitude",
        f"ch{ch}_signal_energy",
        f"ch{ch}_slope_energy",
        f"ch{ch}_max_abs_slope",
        f"ch{ch}_high_freq_power",
        f"ch{ch}_low_freq_power",
        f"ch{ch}_high_low_ratio",
        f"ch{ch}_zero_crossing_rate",
        f"ch{ch}_peak_density",
    ]
FEATURE_NAMES += ["inter_lead_correlation"]
FEATURE_DIM = len(FEATURE_NAMES)          # 21
EPS = 1e-6
_HF_CUT_FRAC = 0.2                        # split low/high freq at 20% of nyquist

# --------------------------- numpy ---------------------------
def _per_channel_features_np(c):
    # c: (N, L)
    amp_range = c.max(axis=1) - c.min(axis=1)
    max_abs   = np.abs(c).max(axis=1)
    energy    = np.mean(c ** 2, axis=1)
    d         = np.diff(c, axis=1)
    slope_en  = np.mean(d ** 2, axis=1)
    max_slope = np.abs(d).max(axis=1)
    Fc        = np.fft.rfft(c, axis=1)
    power     = Fc.real ** 2 + Fc.imag ** 2
    nbins     = power.shape[1]
    cut       = max(2, int(_HF_CUT_FRAC * (nbins - 1)))
    low_p     = np.mean(power[:, 1:cut], axis=1)
    high_p    = np.mean(power[:, cut:], axis=1)
    ratio     = high_p / (low_p + EPS)
    s         = np.sign(c)
    zcr       = np.mean(np.abs(np.diff(s, axis=1)) > 0, axis=1)
    left      = c[:, 1:-1] > c[:, :-2]
    right     = c[:, 1:-1] > c[:, 2:]
    thr       = (c.mean(axis=1) + c.std(axis=1))[:, None]
    ispeak    = left & right & (c[:, 1:-1] > thr)
    peak_den  = ispeak.mean(axis=1)
    return np.stack([amp_range, max_abs, energy, slope_en, max_slope,
                     high_p, low_p, ratio, zcr, peak_den], axis=1)

def extract_risk_features_numpy(X):
    X = np.asarray(X, dtype=np.float32)
    assert X.ndim == 3 and X.shape[1] == 2, f"expected (N,2,L), got {X.shape}"
    c1, c2 = X[:, 0, :], X[:, 1, :]
    f1, f2 = _per_channel_features_np(c1), _per_channel_features_np(c2)
    c1m = c1 - c1.mean(axis=1, keepdims=True)
    c2m = c2 - c2.mean(axis=1, keepdims=True)
    num = np.sum(c1m * c2m, axis=1)
    den = np.sqrt(np.sum(c1m ** 2, axis=1) * np.sum(c2m ** 2, axis=1)) + EPS
    corr = (num / den)[:, None]
    return np.concatenate([f1, f2, corr], axis=1).astype(np.float32)

# --------------------------- torch ---------------------------
def _per_channel_features_torch(c):
    # c: (B, L) ; differentiable where it matters (amplitude/energy/slope/FFT)
    amp_range = c.max(dim=1).values - c.min(dim=1).values
    max_abs   = c.abs().max(dim=1).values
    energy    = (c ** 2).mean(dim=1)
    d         = c[:, 1:] - c[:, :-1]
    slope_en  = (d ** 2).mean(dim=1)
    max_slope = d.abs().max(dim=1).values
    Fc        = torch.fft.rfft(c, dim=1)
    power     = Fc.real ** 2 + Fc.imag ** 2
    nbins     = power.shape[1]
    cut       = max(2, int(_HF_CUT_FRAC * (nbins - 1)))
    low_p     = power[:, 1:cut].mean(dim=1)
    high_p    = power[:, cut:].mean(dim=1)
    ratio     = high_p / (low_p + EPS)
    s         = torch.tanh(20.0 * c)                       # smooth sign surrogate
    zcr       = (s[:, 1:] - s[:, :-1]).abs().mean(dim=1) * 0.5
    left      = (c[:, 1:-1] - c[:, :-2])
    right     = (c[:, 1:-1] - c[:, 2:])
    thr       = c.mean(dim=1, keepdim=True) + c.std(dim=1, keepdim=True)
    ispeak    = ((left > 0) & (right > 0) & (c[:, 1:-1] > thr)).float()
    peak_den  = ispeak.mean(dim=1)
    return torch.stack([amp_range, max_abs, energy, slope_en, max_slope,
                        high_p, low_p, ratio, zcr, peak_den], dim=1)

def extract_risk_features_torch(x):
    # x: (B, 2, L)
    assert x.dim() == 3 and x.shape[1] == 2, f"expected (B,2,L), got {tuple(x.shape)}"
    c1, c2 = x[:, 0, :], x[:, 1, :]
    f1, f2 = _per_channel_features_torch(c1), _per_channel_features_torch(c2)
    c1m = c1 - c1.mean(dim=1, keepdim=True)
    c2m = c2 - c2.mean(dim=1, keepdim=True)
    num = (c1m * c2m).sum(dim=1)
    den = torch.sqrt((c1m ** 2).sum(dim=1) * (c2m ** 2).sum(dim=1)) + EPS
    corr = (num / den).unsqueeze(1)
    return torch.cat([f1, f2, corr], dim=1)

def compute_feature_stats(features, labels):
    feature_mean = features.mean(axis=0)
    feature_std  = features.std(axis=0) + 1e-6
    dmask, nmask = (labels == 1), (labels == 0)
    dangerous_feature_mean = features[dmask].mean(axis=0) if dmask.any() else feature_mean.copy()
    normal_feature_mean    = features[nmask].mean(axis=0) if nmask.any() else feature_mean.copy()
    return (feature_mean.astype(np.float32), feature_std.astype(np.float32),
            dangerous_feature_mean.astype(np.float32), normal_feature_mean.astype(np.float32))

# sanity check: numpy vs torch agree on a small batch
_chk = X[:4]
_np = extract_risk_features_numpy(_chk)
_to = extract_risk_features_torch(torch.from_numpy(_chk)).numpy()
print("FEATURE_DIM =", FEATURE_DIM)
print("numpy/torch max abs diff (sanity):", float(np.abs(_np - _to).max()))
print("feature names:", FEATURE_NAMES)

## 7. Dataset & DataLoaders
We split into train / val, compute **dataset-level** ECG mean/std and risk-feature
mean/std **from the training set only**, then standardize. We deliberately use
mean/std (not per-sample min-max) because absolute amplitude is clinically
meaningful.

In [ ]:
class ECGDataset(torch.utils.data.Dataset):
    def __init__(self, X, y, risk_features, mean, std):
        # X: (N,2,L) raw ; risk_features: (N,F) ALREADY normalized
        # mean/std: ECG per-channel stats, shape (2,1)
        self.X = torch.from_numpy(np.asarray(X, np.float32))
        self.y = torch.from_numpy(np.asarray(y, np.int64))
        self.rf = torch.from_numpy(np.asarray(risk_features, np.float32))
        self.mean = torch.as_tensor(mean, dtype=torch.float32)
        self.std  = torch.as_tensor(std, dtype=torch.float32)
    def __len__(self):
        return self.X.shape[0]
    def __getitem__(self, i):
        x = (self.X[i] - self.mean) / self.std
        return {"x": x, "label": self.y[i], "risk_features": self.rf[i]}

# ---- stratified train / val split (개선 3) ----
_y = np.asarray(y)
idx0 = np.where(_y == 0)[0]
idx1 = np.where(_y == 1)[0]
_rng = np.random.default_rng(CONFIG["train"]["seed"])
_rng.shuffle(idx0); _rng.shuffle(idx1)
n0_train = int(len(idx0) * CONFIG["data"]["train_ratio"])
n1_train = int(len(idx1) * CONFIG["data"]["train_ratio"])
tr_idx = np.concatenate([idx0[:n0_train], idx1[:n1_train]])
va_idx = np.concatenate([idx0[n0_train:], idx1[n1_train:]])
_rng.shuffle(tr_idx); _rng.shuffle(va_idx)
X_tr, y_tr = X[tr_idx], y[tr_idx]
X_va, y_va = X[va_idx], y[va_idx]

# ---- experiment-mode subset (개선 8): true_only -> y==1 만 ----
EXP_MODE = CONFIG["experiment"]["mode"]
if EXP_MODE == "true_only":
    _mt = (y_tr == 1); _mv = (y_va == 1)
    X_tr, y_tr = X_tr[_mt], y_tr[_mt]
    X_va, y_va = X_va[_mv], y_va[_mv]
    print("[mode=true_only] 학습/검증을 true-alarm(y==1) 샘플로 제한")

# ---- risk features (computed on RAW ECG) + train-only stats ----
rf_tr_raw = extract_risk_features_numpy(X_tr)
rf_va_raw = extract_risk_features_numpy(X_va)
feature_mean, feature_std, dangerous_feature_mean, normal_feature_mean = \
    compute_feature_stats(rf_tr_raw, y_tr)
rf_tr = (rf_tr_raw - feature_mean) / feature_std
rf_va = (rf_va_raw - feature_mean) / feature_std

# ---- ECG normalization stats (train only, per-channel) ----
ecg_mean = X_tr.mean(axis=(0, 2)).astype(np.float32).reshape(2, 1)   # (2,1)
ecg_std  = (X_tr.std(axis=(0, 2)).astype(np.float32) + 1e-6).reshape(2, 1)

train_ds = ECGDataset(X_tr, y_tr, rf_tr, ecg_mean, ecg_std)
val_ds   = ECGDataset(X_va, y_va, rf_va, ecg_mean, ecg_std)

_nw = 0   # num_workers=0: worker 프로세스 미사용 -> 멀티프로세싱 cleanup 경고 제거 (속도 영향 거의 없음)

# ---- balanced sampling (개선 4): WeightedRandomSampler ----
_use_sampler = (bool(CONFIG["train"].get("use_weighted_sampler", False))
                and EXP_MODE != "true_only" and len(np.unique(y_tr)) > 1)
if _use_sampler:
    from torch.utils.data import WeightedRandomSampler
    _class_counts   = np.bincount(y_tr, minlength=2)
    _class_weights  = 1.0 / np.maximum(_class_counts, 1)
    _sample_weights = _class_weights[y_tr]
    _sampler = WeightedRandomSampler(
        weights=torch.DoubleTensor(_sample_weights),
        num_samples=len(_sample_weights), replacement=True)
    train_loader = torch.utils.data.DataLoader(
        train_ds, batch_size=CONFIG["data"]["batch_size"], sampler=_sampler,
        shuffle=False, num_workers=_nw, drop_last=True, pin_memory=(DEVICE == "cuda"))
    print("weighted sampling : ENABLED  | class_counts =", _class_counts.tolist())
else:
    train_loader = torch.utils.data.DataLoader(
        train_ds, batch_size=CONFIG["data"]["batch_size"], shuffle=True,
        num_workers=_nw, drop_last=True, pin_memory=(DEVICE == "cuda"))
    print("weighted sampling : DISABLED | shuffle=True")
val_loader = torch.utils.data.DataLoader(
    val_ds, batch_size=CONFIG["data"]["batch_size"], shuffle=False,
    num_workers=_nw, drop_last=False, pin_memory=(DEVICE == "cuda"))

# torch tensors for (de)normalization later
ecg_mean_t = torch.tensor(ecg_mean, dtype=torch.float32, device=DEVICE)   # (2,1)
ecg_std_t  = torch.tensor(ecg_std,  dtype=torch.float32, device=DEVICE)
def denorm_ecg(x):
    return x * ecg_std_t + ecg_mean_t

print("train samples:", len(train_ds), "| val samples:", len(val_ds))
print("train normal/dangerous:", int((y_tr == 0).sum()), "/", int((y_tr == 1).sum()))
print("val   normal/dangerous:", int((y_va == 0).sum()), "/", int((y_va == 1).sum()))
print("ecg_mean:", ecg_mean.ravel(), "| ecg_std:", ecg_std.ravel())
print("risk feature dim:", rf_tr.shape[1])
print("-" * 60)
print("[stratified split report]  train_ratio =", CONFIG["data"]["train_ratio"])
print("Train false count :", int((y_tr == 0).sum()), "| Train true count :", int((y_tr == 1).sum()))
print("Val   false count :", int((y_va == 0).sum()), "| Val   true count :", int((y_va == 1).sum()))
print("Train true ratio  : %.4f" % (float((y_tr == 1).mean()) if len(y_tr) else 0.0))
print("Val   true ratio  : %.4f" % (float((y_va == 1).mean()) if len(y_va) else 0.0))

## 8. Model building blocks
`ResBlock1D`, `DownBlock1D`, `UpBlock1D` built from `Conv1d` + `GroupNorm` + `SiLU`
with residual connections and dilation support. Decoder upsampling uses
`Upsample + Conv1d` (no `ConvTranspose1d`).

In [ ]:
import torch.nn as nn
import torch.nn.functional as Fnn

def gn(num_channels, max_groups=8):
    """GroupNorm with a group count that always divides num_channels."""
    g = math.gcd(int(num_channels), int(max_groups))
    return nn.GroupNorm(max(1, g), num_channels)

class ResBlock1D(nn.Module):
    def __init__(self, in_ch, out_ch, dilation=1):
        super().__init__()
        pad = dilation
        self.conv1 = nn.Conv1d(in_ch, out_ch, 3, padding=pad, dilation=dilation)
        self.norm1 = gn(out_ch)
        self.conv2 = nn.Conv1d(out_ch, out_ch, 3, padding=pad, dilation=dilation)
        self.norm2 = gn(out_ch)
        self.act   = nn.SiLU()
        self.skip  = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    def forward(self, x):
        h = self.act(self.norm1(self.conv1(x)))
        h = self.norm2(self.conv2(h))
        return self.act(h + self.skip(x))

class DownBlock1D(nn.Module):
    """Halve the length (stride-2 conv) then refine with a ResBlock."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.down = nn.Conv1d(in_ch, out_ch, 4, stride=2, padding=1)
        self.norm = gn(out_ch)
        self.act  = nn.SiLU()
        self.res  = ResBlock1D(out_ch, out_ch)
    def forward(self, x):
        x = self.act(self.norm(self.down(x)))
        return self.res(x)

class UpBlock1D(nn.Module):
    """Double the length (linear Upsample + Conv1d) then refine with a ResBlock."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up   = nn.Upsample(scale_factor=2, mode="linear", align_corners=False)
        self.conv = nn.Conv1d(in_ch, out_ch, 3, padding=1)
        self.norm = gn(out_ch)
        self.act  = nn.SiLU()
        self.res  = ResBlock1D(out_ch, out_ch)
    def forward(self, x):
        x = self.up(x)
        x = self.act(self.norm(self.conv(x)))
        return self.res(x)

# quick shape check
_b = torch.zeros(2, 32, 100)
print("ResBlock1D :", tuple(ResBlock1D(32, 64, dilation=2)(_b).shape))
print("DownBlock1D:", tuple(DownBlock1D(32, 64)(_b).shape))
print("UpBlock1D  :", tuple(UpBlock1D(32, 16)(_b).shape))

## 9. Risk-aware CVAE model
Two latent spaces (**shared** + **risk-specific**). The encoder is a 1D-CNN with
downsampling, residual and **dilated** residual blocks `[1,2,4,8]`. The decoder
concatenates `z_shared, z_risk, label_embedding, risk_feature_embedding`, projects,
reshapes, upsamples back to length 2500, and crops/pads to exactly 2500.

In [ ]:
class RiskAwareECGCVAE(nn.Module):
    def __init__(self, in_ch=2, length=2500, base_channels=32,
                 shared_latent_dim=64, risk_latent_dim=64,
                 label_emb_dim=16, feature_emb_dim=64, feature_dim=21,
                 num_classes=2, final_activation=None):
        super().__init__()
        self.length = length
        self.shared_latent_dim = shared_latent_dim
        self.risk_latent_dim = risk_latent_dim
        self.final_activation = final_activation
        ch = [base_channels, base_channels * 2, base_channels * 4, base_channels * 8]

        # ---------- encoder ----------
        self.stem  = nn.Conv1d(in_ch, ch[0], 7, padding=3)
        self.down1 = DownBlock1D(ch[0], ch[0])
        self.down2 = DownBlock1D(ch[0], ch[1])
        self.down3 = DownBlock1D(ch[1], ch[2])
        self.down4 = DownBlock1D(ch[2], ch[3])
        self.dil   = nn.ModuleList([ResBlock1D(ch[3], ch[3], dilation=d) for d in [1, 2, 4, 8]])

        # infer the flattened encoder size with a dummy pass
        with torch.no_grad():
            d = self._encode_conv(torch.zeros(1, in_ch, length))
            self.enc_ch, self.enc_len = d.shape[1], d.shape[2]
        flat = self.enc_ch * self.enc_len

        # condition projection for the ENCODER (개선 5: q(z|x,c) 대신 q(z|x))
        self.enc_cond_proj = nn.Linear(label_emb_dim + feature_emb_dim, self.enc_ch)

        self.fc_mu_shared     = nn.Linear(flat, shared_latent_dim)
        self.fc_logvar_shared = nn.Linear(flat, shared_latent_dim)
        self.fc_mu_risk       = nn.Linear(flat, risk_latent_dim)
        self.fc_logvar_risk   = nn.Linear(flat, risk_latent_dim)

        # ---------- conditioning ----------
        self.label_emb = nn.Embedding(num_classes, label_emb_dim)
        self.feat_emb  = nn.Sequential(
            nn.Linear(feature_dim, feature_emb_dim), nn.SiLU(),
            nn.Linear(feature_emb_dim, feature_emb_dim), nn.SiLU())

        # ---------- decoder ----------
        dec_in = shared_latent_dim + risk_latent_dim + label_emb_dim + feature_emb_dim
        self.dec_fc = nn.Linear(dec_in, flat)
        self.up1 = UpBlock1D(ch[3], ch[2])
        self.up2 = UpBlock1D(ch[2], ch[1])
        self.up3 = UpBlock1D(ch[1], ch[0])
        self.up4 = UpBlock1D(ch[0], ch[0])
        self.head = nn.Conv1d(ch[0], in_ch, 3, padding=1)

    def _encode_conv(self, x):
        x = self.stem(x)
        x = self.down1(x); x = self.down2(x); x = self.down3(x); x = self.down4(x)
        for m in self.dil:
            x = m(x)
        return x

    def encode(self, x, label, risk_features):
        # 개선 5: 인코더도 조건 c=(label_emb, feat_emb) 를 받는다 -> q(z | x, c)
        d  = self._encode_conv(x)                                  # (B, enc_ch, enc_len)
        le = self.label_emb(label)                                 # (B, label_emb_dim)
        fe = self.feat_emb(risk_features)                          # (B, feature_emb_dim)
        cproj = self.enc_cond_proj(torch.cat([le, fe], dim=1))     # (B, enc_ch)
        d = d + cproj.unsqueeze(-1)                                # 시간축으로 broadcast 후 더함
        h = torch.flatten(d, 1)
        return (self.fc_mu_shared(h), self.fc_logvar_shared(h),
                self.fc_mu_risk(h),   self.fc_logvar_risk(h))

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + torch.randn_like(std) * std

    def _fix_len(self, x, L):
        if x.shape[-1] == L:
            return x
        if x.shape[-1] > L:
            return x[..., :L]
        return Fnn.pad(x, (0, L - x.shape[-1]), mode="replicate")

    def decode(self, z_shared, z_risk, label, risk_features):
        le = self.label_emb(label)
        fe = self.feat_emb(risk_features)
        z  = torch.cat([z_shared, z_risk, le, fe], dim=1)
        h  = self.dec_fc(z).view(-1, self.enc_ch, self.enc_len)
        h  = self.up1(h); h = self.up2(h); h = self.up3(h); h = self.up4(h)
        x  = self._fix_len(self.head(h), self.length)
        if self.final_activation == "tanh":
            x = torch.tanh(x)
        return x

    def forward(self, x, label, risk_features):
        mu_s, lv_s, mu_r, lv_r = self.encode(x, label, risk_features)
        z_s = self.reparameterize(mu_s, lv_s)
        z_r = self.reparameterize(mu_r, lv_r)
        x_hat = self.decode(z_s, z_r, label, risk_features)
        return {"x_hat": x_hat, "mu_shared": mu_s, "logvar_shared": lv_s,
                "mu_risk": mu_r, "logvar_risk": lv_r, "z_shared": z_s, "z_risk": z_r}

    @torch.no_grad()
    def generate(self, label, risk_features, num_samples=None):
        device = next(self.parameters()).device
        label = torch.as_tensor(label, device=device)
        if label.dim() == 0:
            num_samples = num_samples or 1
            label = label.repeat(num_samples)
        num_samples = label.shape[0]
        risk_features = torch.as_tensor(risk_features, dtype=torch.float32, device=device)
        if risk_features.dim() == 1:
            risk_features = risk_features.unsqueeze(0).repeat(num_samples, 1)
        z_s = torch.randn(num_samples, self.shared_latent_dim, device=device)
        z_r = torch.randn(num_samples, self.risk_latent_dim, device=device)
        return self.decode(z_s, z_r, label, risk_features)

# build + shape check
_m = RiskAwareECGCVAE(
    in_ch=CONFIG["data"]["input_channels"], length=CONFIG["data"]["input_length"],
    base_channels=CONFIG["model"]["base_channels"],
    shared_latent_dim=CONFIG["model"]["shared_latent_dim"],
    risk_latent_dim=CONFIG["model"]["risk_latent_dim"],
    label_emb_dim=CONFIG["model"]["label_emb_dim"],
    feature_emb_dim=CONFIG["model"]["feature_emb_dim"],
    feature_dim=FEATURE_DIM, final_activation=CONFIG["model"]["final_activation"])
_xb = torch.zeros(2, 2, 2500); _lb = torch.zeros(2, dtype=torch.long); _fb = torch.zeros(2, FEATURE_DIM)
_out = _m(_xb, _lb, _fb)
print("encoder bottleneck (ch,len):", (_m.enc_ch, _m.enc_len))
print("x_hat:", tuple(_out["x_hat"].shape), "| params:", sum(p.numel() for p in _m.parameters()))
del _m, _out

## 10. Auxiliary risk classifier
A small 1D-CNN that maps `(B, 2, 2500) -> (B, 2)`. It is trained jointly with the
CVAE so that generated ECG keeps its intended label identity (normal vs dangerous).

In [ ]:
class RiskClassifier1D(nn.Module):
    def __init__(self, in_ch=2, base=32, num_classes=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_ch, base, 7, stride=2, padding=3),     gn(base),     nn.SiLU(),
            nn.Conv1d(base, base * 2, 5, stride=2, padding=2),  gn(base * 2), nn.SiLU(),
            nn.Conv1d(base * 2, base * 4, 3, stride=2, padding=1), gn(base * 4), nn.SiLU(),
            nn.Conv1d(base * 4, base * 4, 3, stride=2, padding=1), gn(base * 4), nn.SiLU(),
        )
        self.head = nn.Linear(base * 4, num_classes)
    def forward(self, x):
        h = self.net(x).mean(dim=2)          # global average pool over time
        return self.head(h)

_c = RiskClassifier1D()
print("classifier out:", tuple(_c(torch.zeros(2, 2, 2500)).shape),
      "| params:", sum(p.numel() for p in _c.parameters()))
del _c

## 11. Event mask + loss functions
```
total = recon
      + lambda_event      * event_weighted_recon
      + lambda_derivative * derivative_loss
      + lambda_feature    * feature_consistency_loss
      + lambda_classifier * (aux_cls_on_fake + cls_on_real)
      + beta_shared * kl_shared
      + beta_risk   * kl_risk
```
`make_event_mask` flags high-amplitude / high-slope regions and dilates them with
max-pooling, returning a per-time weight map of shape `(B, 1, 2500)`. KL terms are
annealed from 0 up to their max over `kl_anneal_epochs`.

In [ ]:
def make_event_mask(x, event_weight=5.0, amp_k=2.0, slope_k=2.0, pool=15):
    """x: (B,2,L) normalized ECG -> weight map (B,1,L) of 1.0 (calm) .. event_weight (risky)."""
    with torch.no_grad():
        a = x.abs().max(dim=1, keepdim=True).values                 # (B,1,L)
        amp_hot = (a > a.mean(dim=2, keepdim=True) + amp_k * a.std(dim=2, keepdim=True)).float()
        d = (x[..., 1:] - x[..., :-1]).abs().max(dim=1, keepdim=True).values
        d = Fnn.pad(d, (1, 0))                                       # back to length L
        slope_hot = (d > d.mean(dim=2, keepdim=True) + slope_k * d.std(dim=2, keepdim=True)).float()
        hot = torch.clamp(amp_hot + slope_hot, 0.0, 1.0)
        hot = Fnn.max_pool1d(hot, kernel_size=pool, stride=1, padding=pool // 2)
        w = 1.0 + (event_weight - 1.0) * hot
    return w

def kl_divergence(mu, logvar):
    return -0.5 * torch.mean(torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1))

def _flog(f):
    # signed log compresses very different feature magnitudes onto a comparable scale
    return torch.sign(f) * torch.log1p(f.abs())

def compute_cvae_loss(x, out, label, risk_features, classifier,
                      lambda_event=2.0, lambda_derivative=0.5, lambda_feature=0.2,
                      lambda_classifier=0.5, beta_shared=0.0, beta_risk=0.0,
                      event_weight=5.0):
    x_hat = out["x_hat"]

    # plain reconstruction
    recon = Fnn.smooth_l1_loss(x_hat, x)

    # event-weighted reconstruction (focus on risky regions)
    w = make_event_mask(x, event_weight=event_weight)               # (B,1,L)
    elem = Fnn.smooth_l1_loss(x_hat, x, reduction="none")           # (B,2,L)
    event_recon = (w * elem).sum() / (w.sum() * x.shape[1] + 1e-6)

    # derivative loss (preserve sharp slopes)
    dx  = x[..., 1:] - x[..., :-1]
    dxh = x_hat[..., 1:] - x_hat[..., :-1]
    deriv = Fnn.smooth_l1_loss(dxh, dx)

    # differentiable feature-consistency loss
    f_real = extract_risk_features_torch(x)
    f_fake = extract_risk_features_torch(x_hat)
    feat = Fnn.smooth_l1_loss(_flog(f_fake), _flog(f_real))

    # auxiliary classification: keep generated label identity (updates G + classifier),
    # plus classifier trained on real (updates classifier only via detached input)
    aux      = Fnn.cross_entropy(classifier(x_hat), label)
    cls_real = Fnn.cross_entropy(classifier(x.detach()), label)

    # KL on both latent spaces
    kl_s = kl_divergence(out["mu_shared"], out["logvar_shared"])
    kl_r = kl_divergence(out["mu_risk"],   out["logvar_risk"])

    total = (recon
             + lambda_event      * event_recon
             + lambda_derivative * deriv
             + lambda_feature    * feat
             + lambda_classifier * (aux + cls_real)
             + beta_shared * kl_s
             + beta_risk   * kl_r)

    parts = {"total": float(total.detach()), "recon": float(recon.detach()),
             "event": float(event_recon.detach()), "deriv": float(deriv.detach()),
             "feat": float(feat.detach()), "cls": float(aux.detach()),
             "cls_real": float(cls_real.detach()),
             "kl_shared": float(kl_s.detach()), "kl_risk": float(kl_r.detach())}
    return total, parts

print("loss utilities ready.")

## 12. Training & validation loops
AdamW over the CVAE **and** classifier jointly, gradient clipping, tqdm progress,
optional mixed precision, and KL annealing. Every loss component is tracked.

In [ ]:
def _beta(epoch, beta_max, anneal):
    if anneal <= 0:
        return beta_max
    return beta_max * min(1.0, float(epoch) / float(anneal))

def train_one_epoch(model, classifier, loader, optimizer, epoch, cfg, scaler=None):
    model.train(); classifier.train()
    L = cfg["loss"]
    bs = _beta(epoch, L["beta_shared_max"], L["kl_anneal_epochs"])
    br = _beta(epoch, L["beta_risk_max"],   L["kl_anneal_epochs"])
    use_amp = bool(cfg["train"]["use_amp"]) and DEVICE == "cuda"
    params = list(model.parameters()) + list(classifier.parameters())
    agg = {}
    pbar = tqdm(loader, desc=f"train ep{epoch}", leave=False)
    for batch in pbar:
        x  = batch["x"].to(DEVICE)
        lb = batch["label"].to(DEVICE)
        rf = batch["risk_features"].to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        if use_amp:
            with torch.cuda.amp.autocast():
                out = model(x, lb, rf)
                total, parts = compute_cvae_loss(
                    x, out, lb, rf, classifier,
                    lambda_event=L["lambda_event"], lambda_derivative=L["lambda_derivative"],
                    lambda_feature=L["lambda_feature"], lambda_classifier=L["lambda_classifier"],
                    beta_shared=bs, beta_risk=br, event_weight=L["event_weight"])
            scaler.scale(total).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(params, cfg["train"]["grad_clip"])
            scaler.step(optimizer); scaler.update()
        else:
            out = model(x, lb, rf)
            total, parts = compute_cvae_loss(
                x, out, lb, rf, classifier,
                lambda_event=L["lambda_event"], lambda_derivative=L["lambda_derivative"],
                lambda_feature=L["lambda_feature"], lambda_classifier=L["lambda_classifier"],
                beta_shared=bs, beta_risk=br, event_weight=L["event_weight"])
            total.backward()
            torch.nn.utils.clip_grad_norm_(params, cfg["train"]["grad_clip"])
            optimizer.step()
        for k, v in parts.items():
            agg[k] = agg.get(k, 0.0) + v
        pbar.set_postfix(loss=f"{parts['total']:.3f}", recon=f"{parts['recon']:.3f}",
                         cls=f"{parts['cls']:.3f}")
    n = max(1, len(loader))
    return {k: v / n for k, v in agg.items()}, (bs, br)

@torch.no_grad()
def validate_one_epoch(model, classifier, loader, epoch, cfg):
    model.eval(); classifier.eval()
    L = cfg["loss"]; agg = {}
    _correct = 0; _total = 0
    for batch in loader:
        x  = batch["x"].to(DEVICE)
        lb = batch["label"].to(DEVICE)
        rf = batch["risk_features"].to(DEVICE)
        out = model(x, lb, rf)
        _, parts = compute_cvae_loss(
            x, out, lb, rf, classifier,
            lambda_event=L["lambda_event"], lambda_derivative=L["lambda_derivative"],
            lambda_feature=L["lambda_feature"], lambda_classifier=L["lambda_classifier"],
            beta_shared=L["beta_shared_max"], beta_risk=L["beta_risk_max"],
            event_weight=L["event_weight"])
        for k, v in parts.items():
            agg[k] = agg.get(k, 0.0) + v
        _pred = classifier(x).argmax(dim=1)               # 개선 6: 실제 ECG 분류 정확도
        _correct += int((_pred == lb).sum()); _total += int(lb.numel())
    n = max(1, len(loader))
    res = {k: v / n for k, v in agg.items()}
    res["cls_acc"] = _correct / max(1, _total)
    return res

print("train/validate functions ready.")

## 13. Main training execution
Instantiates the model + classifier + AdamW, trains for `CONFIG["train"]["epochs"]`,
saves `best.pt` / `last.pt`, periodically renders **real / reconstructed / generated**
panels for both classes into `.../samples`, and writes `loss_curve.png`.

In [ ]:
CKPT_DIR    = os.path.join(OUTPUT_DIR, "checkpoints")
SAMPLE_DIR  = os.path.join(OUTPUT_DIR, "samples")
BEST_CKPT   = os.path.join(CKPT_DIR, "best.pt")
LAST_CKPT   = os.path.join(CKPT_DIR, "last.pt")

# normalized class-mean conditioning vectors (used for generation previews)
normal_feat_norm    = (normal_feature_mean    - feature_mean) / feature_std
dangerous_feat_norm = (dangerous_feature_mean - feature_mean) / feature_std

def _pick_ref(Xa, ya, lab):
    idx = np.where(ya == lab)[0]
    return int(idx[0]) if len(idx) else int(np.where(ya == ya[0])[0][0])

_ref_n = _pick_ref(X_va, y_va, 0)
_ref_d = _pick_ref(X_va, y_va, 1)

def _norm_x(raw):                      # raw (2,L) -> normalized tensor (2,L)
    t = torch.from_numpy(np.asarray(raw, np.float32))
    return (t - torch.from_numpy(ecg_mean)) / torch.from_numpy(ecg_std)

def _recon(model, raw, lab):
    model.eval()
    with torch.no_grad():
        x = _norm_x(raw).unsqueeze(0).to(DEVICE)
        rf = (extract_risk_features_numpy(raw[None]) - feature_mean) / feature_std
        rf = torch.from_numpy(rf.astype(np.float32)).to(DEVICE)
        lb = torch.tensor([lab], dtype=torch.long, device=DEVICE)
        xh = model(x, lb, rf)["x_hat"]
        return denorm_ecg(xh)[0].cpu().numpy()

def _gen(model, lab, feat_norm):
    model.eval()
    with torch.no_grad():
        rf = torch.from_numpy(feat_norm.astype(np.float32)).to(DEVICE)
        lb = torch.tensor([lab], dtype=torch.long, device=DEVICE)
        xh = model.generate(lb, rf, num_samples=1)
        return denorm_ecg(xh)[0].cpu().numpy()

def sample_panels(model, epoch):
    rows = [
        ("real normal",        X_va[_ref_n]),
        ("recon normal",       _recon(model, X_va[_ref_n], 0)),
        ("generated normal",   _gen(model, 0, normal_feat_norm)),
        ("real dangerous",     X_va[_ref_d]),
        ("recon dangerous",    _recon(model, X_va[_ref_d], 1)),
        ("generated dangerous",_gen(model, 1, dangerous_feat_norm)),
    ]
    fig, axes = plt.subplots(6, 1, figsize=(13, 13), sharex=True)
    for ax, (name, sig) in zip(axes, rows):
        ax.plot(sig[0], color="tab:blue", lw=0.8, label="ECG1")
        ax.plot(sig[1], color="tab:red",  lw=0.8, label="ECG2", alpha=0.8)
        ax.set_ylabel(name, fontsize=9); ax.grid(True, alpha=0.3)
    axes[0].legend(loc="upper right", fontsize=8)
    axes[0].set_title(f"epoch {epoch}: real / reconstructed / generated", fontsize=12)
    axes[-1].set_xlabel("time step")
    fig.tight_layout()
    path = os.path.join(SAMPLE_DIR, f"epoch_{epoch:03d}.png")
    fig.savefig(path, dpi=100, bbox_inches="tight"); plt.show(); plt.close(fig)

def save_ckpt(path, model, classifier, optimizer, epoch):
    torch.save({
        "model_state": model.state_dict(),
        "classifier_state": classifier.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "config": CONFIG,
        "epoch": epoch,
        "ecg_mean": ecg_mean, "ecg_std": ecg_std,
        "feature_mean": feature_mean, "feature_std": feature_std,
        "dangerous_feature_mean": dangerous_feature_mean,
        "normal_feature_mean": normal_feature_mean,
        "feature_dim": FEATURE_DIM, "feature_names": FEATURE_NAMES,
    }, path)

# ---------------- instantiate ----------------
set_seed(CONFIG["train"]["seed"])
model = RiskAwareECGCVAE(
    in_ch=CONFIG["data"]["input_channels"], length=CONFIG["data"]["input_length"],
    base_channels=CONFIG["model"]["base_channels"],
    shared_latent_dim=CONFIG["model"]["shared_latent_dim"],
    risk_latent_dim=CONFIG["model"]["risk_latent_dim"],
    label_emb_dim=CONFIG["model"]["label_emb_dim"],
    feature_emb_dim=CONFIG["model"]["feature_emb_dim"],
    feature_dim=FEATURE_DIM,
    final_activation=CONFIG["model"]["final_activation"]).to(DEVICE)
classifier = RiskClassifier1D(in_ch=CONFIG["data"]["input_channels"],
                              base=CONFIG["model"]["base_channels"]).to(DEVICE)
optimizer = torch.optim.AdamW(
    list(model.parameters()) + list(classifier.parameters()),
    lr=CONFIG["train"]["learning_rate"], weight_decay=CONFIG["train"]["weight_decay"])
_amp_on = bool(CONFIG["train"]["use_amp"]) and DEVICE == "cuda"
try:
    scaler = torch.amp.GradScaler("cuda", enabled=_amp_on)      # torch>=2.4 (deprecation 경고 제거)
except (AttributeError, TypeError):
    scaler = torch.cuda.amp.GradScaler(enabled=_amp_on)

@torch.no_grad()
def gen_true_alarm_prob(n=64):
    # 개선 6: 생성된 label=1 ECG 에 대한 분류기 true-alarm 확률 (1.0 에 가까울수록 위험하게 보임)
    model.eval(); classifier.eval()
    _fn = ((dangerous_feature_mean - feature_mean) / feature_std).astype(np.float32)
    rf = torch.from_numpy(_fn).to(DEVICE).unsqueeze(0).repeat(n, 1)
    lb = torch.full((n,), 1, dtype=torch.long, device=DEVICE)
    xh = model.generate(lb, rf, num_samples=n)            # normalized
    return float(torch.softmax(classifier(xh), dim=1)[:, 1].mean())

history = {"train": [], "val": []}
best_val = float("inf")
EPOCHS = CONFIG["train"]["epochs"]
print(f"Training for {EPOCHS} epochs on {DEVICE} ...")

for epoch in range(1, EPOCHS + 1):
    tr, (bs, br) = train_one_epoch(model, classifier, train_loader, optimizer, epoch, CONFIG, scaler)
    va = validate_one_epoch(model, classifier, val_loader, epoch, CONFIG)
    history["train"].append(tr); history["val"].append(va)
    print(f"[{epoch:3d}/{EPOCHS}] "
          f"train {tr['total']:.3f} (rec {tr['recon']:.3f} ev {tr['event']:.3f} "
          f"der {tr['deriv']:.3f} feat {tr['feat']:.3f} clsR {tr['cls_real']:.3f} clsG {tr['cls']:.3f} "
          f"klS {tr['kl_shared']:.3f} klR {tr['kl_risk']:.3f}) | "
          f"val {va['total']:.3f} acc {va['cls_acc']:.3f} | beta(s,r)=({bs:.4f},{br:.4f})")

    save_ckpt(LAST_CKPT, model, classifier, optimizer, epoch)
    if va["total"] < best_val:
        best_val = va["total"]
        save_ckpt(BEST_CKPT, model, classifier, optimizer, epoch)
        print(f"    -> new best (val {best_val:.3f}) saved to {BEST_CKPT}")
    if epoch % CONFIG["train"]["sample_every"] == 0 or epoch == EPOCHS:
        sample_panels(model, epoch)
        print(f"    [monitor] real-val cls_acc={va['cls_acc']:.3f} | "
              f"generated(label=1) true-alarm prob={gen_true_alarm_prob():.3f}")

# ---------------- loss curves ----------------
def plot_curves(history):
    keys = ["total", "recon", "event", "deriv", "feat", "cls", "kl_shared", "kl_risk"]
    ep = np.arange(1, len(history["train"]) + 1)
    fig, axes = plt.subplots(2, 4, figsize=(18, 8))
    for ax, k in zip(axes.ravel(), keys):
        ax.plot(ep, [h[k] for h in history["train"]], label="train")
        ax.plot(ep, [h[k] for h in history["val"]], label="val")
        ax.set_title(k); ax.set_xlabel("epoch"); ax.grid(True, alpha=0.3); ax.legend(fontsize=8)
    fig.tight_layout()
    path = os.path.join(OUTPUT_DIR, "loss_curve.png")
    fig.savefig(path, dpi=110, bbox_inches="tight"); plt.show(); plt.close(fig)
    print("saved", path)

plot_curves(history)
print("Training complete. best val total =", round(best_val, 4))

## 13.5 (선택) Stage 2 — true-alarm 미세조정 (개선 7)

Stage 1 은 false + true 를 모두 사용해 **일반적인 ECG 구조**를 학습한다.
Stage 2 는 **true-alarm(y==1) 샘플만** 사용해 위험 파형 형태를 **날카롭게(sharpen)** 다듬는다.
`CONFIG["train"]["use_true_finetune"]=True` (또는 `mode="conditional_all_then_true_finetune"`) 일 때만 실행되며,
분류기는 고정(freeze)하고 CVAE 만 미세조정한 뒤 `checkpoints/best_true_finetuned.pt` 로 저장한다.

In [ ]:
# ===== Stage 2: true-alarm 미세조정 (개선 7) =====
if CONFIG["train"]["use_true_finetune"]:
    BEST_FT_CKPT = os.path.join(CKPT_DIR, "best_true_finetuned.pt")
    _ft_mask_tr = (y_tr == 1); _ft_mask_va = (y_va == 1)
    print(f"[finetune] true train={int(_ft_mask_tr.sum())} | true val={int(_ft_mask_va.sum())}")
    if int(_ft_mask_tr.sum()) == 0:
        print("[finetune] true-alarm 학습 샘플이 없어 건너뜀.")
    else:
        ft_train_ds = ECGDataset(X_tr[_ft_mask_tr], y_tr[_ft_mask_tr], rf_tr[_ft_mask_tr], ecg_mean, ecg_std)
        ft_train_loader = torch.utils.data.DataLoader(
            ft_train_ds, batch_size=CONFIG["data"]["batch_size"], shuffle=True,
            num_workers=0, drop_last=False, pin_memory=(DEVICE == "cuda"))
        if int(_ft_mask_va.sum()) > 0:
            ft_val_ds = ECGDataset(X_va[_ft_mask_va], y_va[_ft_mask_va], rf_va[_ft_mask_va], ecg_mean, ecg_std)
            ft_val_loader = torch.utils.data.DataLoader(
                ft_val_ds, batch_size=CONFIG["data"]["batch_size"], shuffle=False,
                num_workers=0, drop_last=False, pin_memory=(DEVICE == "cuda"))
        else:
            ft_val_loader = ft_train_loader

        # Stage 1 best 에서 시작
        _ck = torch.load(BEST_CKPT, map_location=DEVICE, weights_only=False)
        model.load_state_dict(_ck["model_state"]); classifier.load_state_dict(_ck["classifier_state"])

        # 분류기 고정 -> CVAE 만 미세조정 (분류기 퇴화 방지)
        for _p in classifier.parameters():
            _p.requires_grad_(False)
        ft_opt = torch.optim.AdamW(model.parameters(),
                                   lr=CONFIG["train"]["true_finetune_lr"],
                                   weight_decay=CONFIG["train"]["weight_decay"])
        L = CONFIG["loss"]; K = int(CONFIG["train"]["true_finetune_epochs"])
        best_ft = float("inf")
        print(f"[finetune] {K} epochs, lr={CONFIG['train']['true_finetune_lr']} (label 고정=1, classifier frozen)")
        for ep in range(1, K + 1):
            model.train()
            for batch in ft_train_loader:
                x  = batch["x"].to(DEVICE); lb = batch["label"].to(DEVICE); rf = batch["risk_features"].to(DEVICE)
                ft_opt.zero_grad(set_to_none=True)
                out = model(x, lb, rf)
                total, _ = compute_cvae_loss(
                    x, out, lb, rf, classifier,
                    lambda_event=L["lambda_event"], lambda_derivative=L["lambda_derivative"],
                    lambda_feature=L["lambda_feature"], lambda_classifier=L["lambda_classifier"],
                    beta_shared=L["beta_shared_max"], beta_risk=L["beta_risk_max"],
                    event_weight=L["event_weight"])
                total.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["train"]["grad_clip"])
                ft_opt.step()
            va = validate_one_epoch(model, classifier, ft_val_loader, ep, CONFIG)
            print(f"  [ft {ep:2d}/{K}] val total {va['total']:.3f} | val cls_acc {va['cls_acc']:.3f}")
            if va["total"] < best_ft:
                best_ft = va["total"]
                save_ckpt(BEST_FT_CKPT, model, classifier, ft_opt, ep)
                print(f"      -> new best finetuned (val {best_ft:.3f}) saved to {BEST_FT_CKPT}")
        for _p in classifier.parameters():
            _p.requires_grad_(True)
        print("[finetune] done ->", BEST_FT_CKPT)
else:
    print("Stage 2 true-alarm finetune: SKIPPED (use_true_finetune=False)")

## 14. Load a checkpoint
Rebuilds the model + classifier from the stored config and restores all
normalization / feature statistics. Use this to resume from `best.pt` without
re-training.

In [ ]:
# 개선 7: true-finetune 체크포인트가 존재하면 우선 사용
_ft_ckpt = os.path.join(OUTPUT_DIR, "checkpoints", "best_true_finetuned.pt")
if CONFIG["train"]["use_true_finetune"] and os.path.exists(_ft_ckpt):
    CHECKPOINT_PATH = _ft_ckpt
else:
    CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "checkpoints", "best.pt")

def load_checkpoint(checkpoint_path=CHECKPOINT_PATH, map_location=None):
    global ecg_mean, ecg_std, ecg_mean_t, ecg_std_t
    global feature_mean, feature_std, dangerous_feature_mean, normal_feature_mean
    global FEATURE_DIM, FEATURE_NAMES
    map_location = map_location or DEVICE
    ckpt = torch.load(checkpoint_path, map_location=map_location, weights_only=False)
    cfg = ckpt["config"]
    m = RiskAwareECGCVAE(
        in_ch=cfg["data"]["input_channels"], length=cfg["data"]["input_length"],
        base_channels=cfg["model"]["base_channels"],
        shared_latent_dim=cfg["model"]["shared_latent_dim"],
        risk_latent_dim=cfg["model"]["risk_latent_dim"],
        label_emb_dim=cfg["model"]["label_emb_dim"],
        feature_emb_dim=cfg["model"]["feature_emb_dim"],
        feature_dim=ckpt["feature_dim"],
        final_activation=cfg["model"]["final_activation"]).to(map_location)
    c = RiskClassifier1D(in_ch=cfg["data"]["input_channels"],
                         base=cfg["model"]["base_channels"]).to(map_location)
    m.load_state_dict(ckpt["model_state"]);  m.eval()
    c.load_state_dict(ckpt["classifier_state"]); c.eval()

    # restore stats
    ecg_mean = np.asarray(ckpt["ecg_mean"], np.float32)
    ecg_std  = np.asarray(ckpt["ecg_std"],  np.float32)
    ecg_mean_t = torch.tensor(ecg_mean, dtype=torch.float32, device=map_location)
    ecg_std_t  = torch.tensor(ecg_std,  dtype=torch.float32, device=map_location)
    feature_mean = np.asarray(ckpt["feature_mean"], np.float32)
    feature_std  = np.asarray(ckpt["feature_std"],  np.float32)
    dangerous_feature_mean = np.asarray(ckpt["dangerous_feature_mean"], np.float32)
    normal_feature_mean    = np.asarray(ckpt["normal_feature_mean"], np.float32)
    FEATURE_DIM   = ckpt["feature_dim"]
    FEATURE_NAMES = ckpt["feature_names"]
    print(f"Loaded checkpoint from epoch {ckpt['epoch']} :: {checkpoint_path}")
    return m, c

# load best weights (comment out to keep the just-trained in-memory model)
model, classifier = load_checkpoint(CHECKPOINT_PATH)
print("using checkpoint :", CHECKPOINT_PATH)

## 15. Conditional generation (`risk_strength` knob)
`label=0` conditions on the **normal** feature mean; `label=1` on the **dangerous**
feature mean. Setting `risk_strength > 1.0` amplifies the dangerous descriptors
(amplitude range, max-abs amplitude, slope energy, max-abs slope, high-frequency
power, high/low ratio, peak density) to push the generator toward more extreme,
clearly-risky waveforms. Saves `generated_normal.npy` and `generated_dangerous.npy`.

In [ ]:
# indices of the descriptors that "risk_strength" amplifies
_BOOST_KEYS = ["amplitude_range", "max_abs_amplitude", "slope_energy",
               "max_abs_slope", "high_freq_power", "high_low_ratio", "peak_density"]
BOOST_IDX = [i for i, n in enumerate(FEATURE_NAMES) if any(k in n for k in _BOOST_KEYS)]

def generate_ecg_samples(model, label, num_samples, risk_strength,
                         dangerous_feature_mean, normal_feature_mean,
                         feature_mean, feature_std):
    """Return generated ECG as numpy (num_samples, 2, 2500) in raw amplitude units."""
    base = (dangerous_feature_mean if int(label) == 1 else normal_feature_mean).astype(np.float32).copy()
    if int(label) == 1 and risk_strength != 1.0:
        base[BOOST_IDX] = base[BOOST_IDX] * float(risk_strength)
    feat_norm = (base - feature_mean) / feature_std
    feat_t = torch.from_numpy(feat_norm.astype(np.float32)).to(DEVICE).unsqueeze(0).repeat(num_samples, 1)
    lab_t  = torch.full((num_samples,), int(label), dtype=torch.long, device=DEVICE)
    model.eval()
    with torch.no_grad():
        xh = model.generate(lab_t, feat_t, num_samples=num_samples)   # normalized
        xh = denorm_ecg(xh)                                           # raw units
    return xh.cpu().numpy().astype(np.float32)

GEN_DIR = os.path.join(OUTPUT_DIR, "generated")
DANGER_STRENGTH = float(CONFIG["generate"]["dangerous_risk_strength"])
NORMAL_STRENGTH = float(CONFIG["generate"]["normal_risk_strength"])

# 부족분(deficit) 자동 계산: 분류기 train(train_10s.pt) 의 false-true 차이만큼 dangerous 생성
_n_false = int((y == 0).sum()); _n_true = int((y == 1).sum())
_deficit = max(0, _n_false - _n_true)
if CONFIG["generate"].get("auto_fill_deficit", True):
    N_GEN_DANGER = max(int(CONFIG["generate"]["num_samples"]),
                       _deficit + int(CONFIG["generate"].get("fill_buffer", 64)))
else:
    N_GEN_DANGER = int(CONFIG["generate"]["num_samples"])
N_GEN_NORMAL = min(N_GEN_DANGER, int(CONFIG["generate"]["num_samples"]))   # normal 은 평가용
N_GEN = N_GEN_DANGER                                                       # 요약/로그 호환

print("=== generation settings (개선 9 + deficit 자동충당) ===")
print(f"  real false/true         : {_n_false}/{_n_true}  -> deficit {_deficit}")
print(f"  generate dangerous      : {N_GEN_DANGER}  (auto_fill={CONFIG['generate'].get('auto_fill_deficit', True)}, buffer={CONFIG['generate'].get('fill_buffer', 64)})")
print(f"  generate normal (eval)  : {N_GEN_NORMAL}")
print(f"  dangerous risk_strength : {DANGER_STRENGTH}  (label=1, dangerous_feature_mean)")
print(f"  normal    risk_strength : {NORMAL_STRENGTH}  (label=0, normal_feature_mean)")
print(f"  boosted risk features   : {[FEATURE_NAMES[i] for i in BOOST_IDX]}")

gen_normal = generate_ecg_samples(
    model, label=0, num_samples=N_GEN_NORMAL, risk_strength=NORMAL_STRENGTH,
    dangerous_feature_mean=dangerous_feature_mean, normal_feature_mean=normal_feature_mean,
    feature_mean=feature_mean, feature_std=feature_std)

gen_danger = generate_ecg_samples(
    model, label=1, num_samples=N_GEN_DANGER, risk_strength=DANGER_STRENGTH,
    dangerous_feature_mean=dangerous_feature_mean, normal_feature_mean=normal_feature_mean,
    feature_mean=feature_mean, feature_std=feature_std)

np.save(os.path.join(GEN_DIR, "generated_normal.npy"), gen_normal)
np.save(os.path.join(GEN_DIR, "generated_dangerous.npy"), gen_danger)
print("generated_normal:", gen_normal.shape, "| generated_dangerous:", gen_danger.shape)
print("saved to", GEN_DIR)

for i in range(3):
    plot_ecg_sample(gen_normal[i], label=0, title="GENERATED normal", idx=i)
for i in range(3):
    plot_ecg_sample(gen_danger[i], label=1, title=f"GENERATED dangerous (risk_strength={DANGER_STRENGTH})", idx=i)

## 16. Evaluation
Compare **generated dangerous** ECG against **real dangerous** and **real normal**
across the risk descriptors, the classifier's dangerous probability, and feature
distance. A good dangerous generator should: (1) be classified as dangerous,
(2) sit closer to real-dangerous features than to real-normal features, and
(3) preserve sharp slopes, amplitude bursts and rhythm abnormality.

In [ ]:
EVAL_DIR = os.path.join(OUTPUT_DIR, "evaluation")
os.makedirs(EVAL_DIR, exist_ok=True)
name2idx = {n: i for i, n in enumerate(FEATURE_NAMES)}

# real groups (전체 데이터 기준)
real_false = X[y == 0]
real_true  = X[y == 1]

def _safe_feats(a):
    return extract_risk_features_numpy(a) if len(a) else np.zeros((1, FEATURE_DIM), np.float32)
feat_rf = _safe_feats(real_false)   # real false alarm
feat_rt = _safe_feats(real_true)    # real true alarm
feat_gd = extract_risk_features_numpy(gen_danger)   # generated dangerous
feat_gn = extract_risk_features_numpy(gen_normal)   # generated normal

def _znorm(f): return (f - feature_mean) / feature_std
m_rf = _znorm(feat_rf).mean(0); m_rt = _znorm(feat_rt).mean(0)
m_gd = _znorm(feat_gd).mean(0); m_gn = _znorm(feat_gn).mean(0)

# ---- 4 distances (개선 10) ----
d_gd_rt = float(np.linalg.norm(m_gd - m_rt))   # generated dangerous -> real true
d_gd_rf = float(np.linalg.norm(m_gd - m_rf))   # generated dangerous -> real false
d_gn_rf = float(np.linalg.norm(m_gn - m_rf))   # generated normal    -> real false
d_gn_rt = float(np.linalg.norm(m_gn - m_rt))   # generated normal    -> real true

print("=== feature-space distance (normalized, mean vectors) [개선 10] ===")
print(f"distance(generated dangerous, real true alarm)  = {d_gd_rt:.3f}")
print(f"distance(generated dangerous, real false alarm) = {d_gd_rf:.3f}")
print(f"distance(generated normal,   real false alarm)  = {d_gn_rf:.3f}")
print(f"distance(generated normal,   real true alarm)   = {d_gn_rt:.3f}")
print("criterion d(gen dangerous, real true) < d(gen dangerous, real false) ->",
      "GOOD" if d_gd_rt < d_gd_rf else "needs more training")

# ---- classifier true-alarm probability ----
def _danger_prob(raw_batch):
    if len(raw_batch) == 0:
        return float("nan")
    x = (torch.from_numpy(np.asarray(raw_batch, np.float32)) - torch.from_numpy(ecg_mean)) / torch.from_numpy(ecg_std)
    classifier.eval()
    with torch.no_grad():
        p = torch.softmax(classifier(x.to(DEVICE)), dim=1)[:, 1]
    return float(p.mean())

p_gd = _danger_prob(gen_danger)
p_gn = _danger_prob(gen_normal)
p_rt = _danger_prob(real_true[:min(len(real_true), 256)])
p_rf = _danger_prob(real_false[:min(len(real_false), 256)])
print("\n=== classifier true-alarm probability (1.0 = looks dangerous) ===")
print(f"generated dangerous : {p_gd:.3f}")
print(f"generated normal    : {p_gn:.3f}")
print(f"real true alarm     : {p_rt:.3f}")
print(f"real false alarm    : {p_rf:.3f}")

# ---- distribution histograms (개선 10) ----
sel = ["ch1_amplitude_range", "ch1_slope_energy", "ch1_high_freq_power", "ch1_max_abs_slope"]
fig, axes = plt.subplots(1, len(sel), figsize=(5 * len(sel), 4))
for ax, key in zip(np.atleast_1d(axes), sel):
    j = name2idx[key]
    ax.hist(feat_rf[:, j], bins=30, alpha=0.45, label="real false", density=True)
    ax.hist(feat_rt[:, j], bins=30, alpha=0.45, label="real true",  density=True)
    ax.hist(feat_gd[:, j], bins=30, alpha=0.45, label="gen danger", density=True)
    ax.hist(feat_gn[:, j], bins=30, alpha=0.45, label="gen normal", density=True)
    ax.set_title(key, fontsize=10); ax.legend(fontsize=7); ax.grid(True, alpha=0.3)
fig.tight_layout()
_hp = os.path.join(EVAL_DIR, "feature_histograms.png")
fig.savefig(_hp, dpi=110, bbox_inches="tight"); plt.show(); plt.close(fig)
print("saved", _hp)

# ---- PCA scatter (optional, 개선 10) ----
pca_path = None
try:
    from sklearn.decomposition import PCA
    groups = [("real false", _znorm(feat_rf)), ("real true", _znorm(feat_rt)),
              ("gen danger", _znorm(feat_gd)), ("gen normal", _znorm(feat_gn))]
    pcs = PCA(n_components=2).fit(np.concatenate([g[1] for g in groups], 0))
    fig, ax = plt.subplots(figsize=(7, 6))
    for name, f in groups:
        z = pcs.transform(f)
        ax.scatter(z[:, 0], z[:, 1], s=8, alpha=0.5, label=name)
    ax.set_title("PCA of risk features"); ax.legend(); ax.grid(True, alpha=0.3)
    pca_path = os.path.join(EVAL_DIR, "pca_scatter.png")
    fig.savefig(pca_path, dpi=110, bbox_inches="tight"); plt.show(); plt.close(fig)
    print("saved", pca_path)
except Exception as _e:
    print("PCA skipped (sklearn unavailable?):", _e)

# ---- waveform comparison (개선 10) ----
def _first(a):
    return a[0] if len(a) else np.zeros((2, CONFIG["data"]["input_length"]), np.float32)
rows = [("real true",  _first(real_true)), ("real false", _first(real_false)),
        ("gen danger", gen_danger[0]),     ("gen normal", gen_normal[0])]
fig, axes = plt.subplots(4, 1, figsize=(13, 10), sharex=True)
for ax, (name, sig) in zip(axes, rows):
    ax.plot(sig[0], color="tab:blue", lw=0.8, label="ECG1")
    ax.plot(sig[1], color="tab:red",  lw=0.8, alpha=0.8, label="ECG2")
    ax.set_ylabel(name, fontsize=9); ax.grid(True, alpha=0.3)
axes[0].legend(loc="upper right", fontsize=8)
axes[0].set_title("real true / real false / generated danger / generated normal")
axes[-1].set_xlabel("time step")
fig.tight_layout()
_wp = os.path.join(EVAL_DIR, "waveform_compare.png")
fig.savefig(_wp, dpi=110, bbox_inches="tight"); plt.show(); plt.close(fig)
print("saved", _wp)

# ---- results dict (개선 11 summary 에서 사용) ----
EVAL_RESULTS = {
    "distance_gen_dangerous_to_real_true":  d_gd_rt,
    "distance_gen_dangerous_to_real_false": d_gd_rf,
    "distance_gen_normal_to_real_false":    d_gn_rf,
    "distance_gen_normal_to_real_true":     d_gn_rt,
    "dangerous_closer_to_true":             bool(d_gd_rt < d_gd_rf),
    "clf_prob_gen_dangerous": p_gd, "clf_prob_gen_normal": p_gn,
    "clf_prob_real_true": p_rt, "clf_prob_real_false": p_rf,
    "plots": {"feature_histograms": _hp, "pca_scatter": pca_path, "waveform_compare": _wp},
}
print("\nEVAL_RESULTS ready. evaluation dir:", EVAL_DIR)


## 17. Save & zip outputs
Bundles everything under `/content/risk_ecg_cvae_outputs` into a single zip and
prints the key paths.

In [ ]:
import glob

zip_path = "/content/risk_ecg_cvae_outputs.zip"
get_ipython().system('rm -f ' + zip_path)
get_ipython().system('zip -r -q ' + zip_path + ' ' + OUTPUT_DIR)

print("==================== OUTPUT SUMMARY ====================")
print("best checkpoint   :", BEST_CKPT, "(exists:", os.path.exists(BEST_CKPT), ")")
print("last checkpoint   :", LAST_CKPT, "(exists:", os.path.exists(LAST_CKPT), ")")
print("generated normal  :", os.path.join(GEN_DIR, "generated_normal.npy"))
print("generated danger  :", os.path.join(GEN_DIR, "generated_dangerous.npy"))
print("loss curve        :", os.path.join(OUTPUT_DIR, "loss_curve.png"))
print("feature histograms:", os.path.join(OUTPUT_DIR, "feature_histograms.png"))
print("sample panels     :")
for p in sorted(glob.glob(os.path.join(SAMPLE_DIR, "*.png"))):
    print("    -", p)
print("zipped outputs    :", zip_path, "(exists:", os.path.exists(zip_path), ")")
print("=======================================================")
print("Done. Download the zip from the Colab Files panel (left sidebar).")

In [ ]:
# ===== 최종 요약 + run_summary.json (개선 11) =====
import json as _json
_ft_path  = os.path.join(OUTPUT_DIR, "checkpoints", "best_true_finetuned.pt")
_gen_norm = os.path.join(GEN_DIR, "generated_normal.npy")
_gen_dang = os.path.join(GEN_DIR, "generated_dangerous.npy")

run_summary = {
    "training_mode":        CONFIG["experiment"]["mode"],
    "use_weighted_sampler": bool(CONFIG["train"]["use_weighted_sampler"]),
    "use_true_finetune":    bool(CONFIG["train"]["use_true_finetune"]),
    "class_counts": {
        "train_false": int((y_tr == 0).sum()), "train_true": int((y_tr == 1).sum()),
        "val_false":   int((y_va == 0).sum()), "val_true":   int((y_va == 1).sum()),
    },
    "split_ratio": {
        "train_ratio":      CONFIG["data"]["train_ratio"],
        "train_true_ratio": float((y_tr == 1).mean()) if len(y_tr) else 0.0,
        "val_true_ratio":   float((y_va == 1).mean()) if len(y_va) else 0.0,
    },
    "best_val_total": float(best_val),
    "evaluation":     EVAL_RESULTS,
    "generated":      {"normal": _gen_norm, "dangerous": _gen_dang},
    "checkpoints":    {"best": BEST_CKPT,
                       "best_true_finetuned": _ft_path if os.path.exists(_ft_path) else None},
    "evaluation_dir": EVAL_DIR,
    "config":         CONFIG,
}
_sp = os.path.join(OUTPUT_DIR, "run_summary.json")
with open(_sp, "w") as _f:
    _json.dump(run_summary, _f, indent=2, default=str)

print("==================== RUN SUMMARY ====================")
print("Training mode                    :", run_summary["training_mode"])
print("Use weighted sampler             :", run_summary["use_weighted_sampler"])
print("Use true finetune                :", run_summary["use_true_finetune"])
print("Train false count                :", run_summary["class_counts"]["train_false"])
print("Train true count                 :", run_summary["class_counts"]["train_true"])
print("Validation false count           :", run_summary["class_counts"]["val_false"])
print("Validation true count            :", run_summary["class_counts"]["val_true"])
print("Best checkpoint                  :", run_summary["checkpoints"]["best"])
print("Best true-finetuned checkpoint   :", run_summary["checkpoints"]["best_true_finetuned"])
print("Generated dangerous file         :", run_summary["generated"]["dangerous"])
print("Generated normal file            :", run_summary["generated"]["normal"])
print("Evaluation output directory      :", run_summary["evaluation_dir"])
print("Best val total                   :", round(run_summary["best_val_total"], 4))
print("d(gen danger, real true/false)   :",
      round(EVAL_RESULTS["distance_gen_dangerous_to_real_true"], 3), "/",
      round(EVAL_RESULTS["distance_gen_dangerous_to_real_false"], 3))
print("run_summary.json                 :", _sp)
print("=====================================================")
print()
print("Updated risk_aware_ecg_cvae_colab.ipynb with:")
print("1. stratified split")
print("2. weighted sampling")
print("3. condition-aware encoder")
print("4. stronger label conditioning")
print("5. optional true-only / true-finetune modes")
print("6. true-vs-false generation evaluation")

## 최종 export — Classifier 와 경로/형식 맞춤
생성된 **dangerous(=True alarm)** 샘플을 Classifier 가 로드하는 경로·키에 맞춰 저장.
- 경로: `/content/drive/MyDrive/vtac_project/outputs/cvae_v4/generated_true_v4.pt`
- 키: `X_syn [N,4,2500]`, `y_syn [N]=1`, `m_syn [N,4]=[1,1,0,0]`
- 2채널(ECG1,ECG2) → 4채널 패딩(PPG/ABP=0) + per-window z-정규화 (실데이터 스케일 맞춤)

In [ ]:
# ===== Classifier 용 export (경로/키/형식 일치) =====
from google.colab import drive as _drv
try: _drv.mount('/content/drive')
except Exception as _e: print('drive mount note:', _e)

CLF_OUT_DIR = '/content/drive/MyDrive/vtac_project/outputs/cvae_v4'   # Classifier 의 OUT_DIR 과 동일
os.makedirs(CLF_OUT_DIR, exist_ok=True)

# gen_danger: [N,2,2500] raw -> per-window z-norm -> 4채널 패딩
_g = torch.from_numpy(gen_danger.astype(np.float32))
_mu = _g.mean(dim=-1, keepdim=True); _sd = _g.std(dim=-1, keepdim=True) + 1e-6
_g = (_g - _mu) / _sd
_N, _, _L = _g.shape
X_syn = torch.zeros(_N, 4, _L, dtype=torch.float32); X_syn[:, 0:2] = _g   # ECG1,ECG2 / PPG,ABP=0
y_syn = torch.ones(_N, dtype=torch.long)                                 # True alarm
m_syn = torch.zeros(_N, 4, dtype=torch.float32); m_syn[:, 0:2] = 1.0      # [1,1,0,0]

_path = f'{CLF_OUT_DIR}/generated_true_v4.pt'
torch.save({'X_syn': X_syn, 'y_syn': y_syn, 'm_syn': m_syn,
            'channels': ['ECG1','ECG2','PPG','ABP'], 'method': 'risk_aware_cvae'}, _path)
print('saved for classifier:', _path)
print('  X_syn', tuple(X_syn.shape), '| y_syn', tuple(y_syn.shape), '| m_syn', tuple(m_syn.shape), '| N =', _N)

## 18. PPG / ABP Generation Extension for 4-Channel Classifier Augmentation

The ECG CVAE generates ECG1 and ECG2 only. However, the classifier uses four physiological channels: ECG1, ECG2, PPG, and ABP. This section generates physiologically plausible PPG and ABP channels conditioned on the generated ECG, then exports a complete `(N, 4, 2500)` synthetic true-alarm dataset for the classifier.

In [ ]:
# ===== 18.1 PPG/ABP 확장 설정 =====
import os, numpy as np, torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

PPG_ABP_OUTPUT_DIR = "/content/drive/MyDrive/vtac_project/outputs/cvae_v4"
PPG_ABP_TRAIN_PT   = "/content/drive/MyDrive/vtac_project/data/processed/train_10s.pt"
USE_NEURAL_PULSE_GENERATOR = False    # 기본: 단순 규칙기반 사용 (신경망 코드는 보존, 비활성)
FALLBACK_TO_RULE_BASED     = True
PULSE_GEN_EPOCHS    = 30
PULSE_GEN_BATCH_SIZE = 64
PULSE_GEN_LR        = 2e-4
PULSE_LENGTH        = 2500
FS                  = 250
PULSE_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

try:
    from google.colab import drive as _drv
    _drv.mount("/content/drive")
except Exception as _e:
    print("drive mount note:", _e)

PPG_ABP_EXT_DIR   = os.path.join(PPG_ABP_OUTPUT_DIR, "ppg_abp_extension")
PPG_ABP_PLOTS_DIR = os.path.join(PPG_ABP_EXT_DIR, "plots")
os.makedirs(PPG_ABP_EXT_DIR, exist_ok=True)
os.makedirs(PPG_ABP_PLOTS_DIR, exist_ok=True)
print("PPG/ABP ext dir:", PPG_ABP_EXT_DIR, "| device:", PULSE_DEVICE)

In [ ]:
# ===== 18.2 실제 4채널 데이터 로드 (PPG/ABP 생성기 학습용) =====
REAL4_AVAILABLE = False
X_real_complete = y_real_complete = None
if os.path.exists(PPG_ABP_TRAIN_PT):
    _d = torch.load(PPG_ABP_TRAIN_PT, map_location="cpu", weights_only=False)
    X_real = _d["X"].float()            # (N,4,2500)  ch: 0=ECG1,1=ECG2,2=PPG,3=ABP
    y_real = _d["y"].long()             # (N,)
    m_real = _d["m_channel"].float()    # (N,4)
    complete      = (m_real[:, 0] == 1) & (m_real[:, 1] == 1) & (m_real[:, 2] == 1) & (m_real[:, 3] == 1)
    complete_true = complete & (y_real == 1)
    print("total real samples      :", len(X_real))
    print("complete 4-ch samples   :", int(complete.sum()))
    print("complete true-alarm     :", int(complete_true.sum()))
    if int(complete_true.sum()) >= 50:
        sel = complete_true
        print("  -> true-alarm 충분: true 만 사용")
    else:
        sel = complete
        print("  -> true-alarm 부족: 모든 complete 4-ch 사용")
    X_real_complete = X_real[sel].contiguous()
    y_real_complete = y_real[sel].contiguous()
    print("selected training samples:", len(X_real_complete))
    REAL4_AVAILABLE = len(X_real_complete) >= 16
else:
    print("real 4-ch 파일 없음 -> rule-based 사용 예정:", PPG_ABP_TRAIN_PT)

In [ ]:
# ===== 18.3 ECG-조건부 신경망 PPG/ABP 생성기 =====
class ResDilatedBlock(nn.Module):
    def __init__(self, ch, dilation, k=7, groups=8):
        super().__init__()
        pad = dilation * (k - 1) // 2
        self.conv1 = nn.Conv1d(ch, ch, k, padding=pad, dilation=dilation)
        self.gn1   = nn.GroupNorm(groups, ch)
        self.conv2 = nn.Conv1d(ch, ch, k, padding=pad, dilation=dilation)
        self.gn2   = nn.GroupNorm(groups, ch)
        self.act   = nn.SiLU()
    def forward(self, x):
        h = self.act(self.gn1(self.conv1(x)))
        h = self.gn2(self.conv2(h))
        return self.act(x + h)              # residual; length preserved

class ECGToPulseGenerator(nn.Module):
    """ecg (B,2,2500) -> pulse_hat (B,2,2500): ch0=PPG, ch1=ABP. 길이 정확히 2500 보존."""
    def __init__(self, in_ch=2, out_ch=2, base=64, dilations=(1, 2, 4, 8, 16)):
        super().__init__()
        self.inp = nn.Conv1d(in_ch, base, 7, padding=3)
        self.act = nn.SiLU()
        self.blocks = nn.ModuleList([ResDilatedBlock(base, d) for d in dilations])
        self.mid = nn.Conv1d(base, base, 7, padding=3)
        self.out = nn.Conv1d(base, out_ch, 7, padding=3)
    def forward(self, ecg):
        h = self.act(self.inp(ecg))
        for b in self.blocks:
            h = b(h)
        h = self.act(self.mid(h))
        return self.out(h)

print("ECGToPulseGenerator defined.")

In [ ]:
# ===== 18.4 미분 가능한 ECG-pulse coupling 유틸 =====
def _zscore_t(x, eps=1e-6):
    return (x - x.mean(dim=-1, keepdim=True)) / (x.std(dim=-1, keepdim=True) + eps)

def signal_envelope_torch(x, pool=25):
    # x: (B,C,L) -> (B,1,L) smoothed event envelope: |x| + |dx|, max over channels, avg-pooled
    a  = x.abs().amax(dim=1, keepdim=True)
    dx = F.pad((x[..., 1:] - x[..., :-1]).abs().amax(dim=1, keepdim=True), (1, 0))
    env = F.avg_pool1d(a + dx, kernel_size=pool, stride=1, padding=pool // 2)
    return env

def best_shift_corr_torch(ecg, pulse, min_lag=25, max_lag=75):
    # ecg: (B,2,L), pulse: (B,1,L) single channel. Return (B,) max corr over physiologic lags.
    env = _zscore_t(signal_envelope_torch(ecg))     # (B,1,L)
    p   = _zscore_t(pulse)                           # (B,1,L)
    L = env.shape[-1]
    best = None
    for lag in range(min_lag, max_lag + 1):
        c = (env[..., :L - lag] * p[..., lag:]).mean(dim=-1).squeeze(1)   # (B,)
        best = c if best is None else torch.maximum(best, c)
    return best

def ecg_pulse_coupling_loss(ecg_real, pulse_real, ecg_fake, pulse_fake):
    # pulse_*: (B,2,L) [PPG, ABP]. match real/fake ECG->pulse coupling strength per channel.
    loss = 0.0
    for ch in range(2):
        cr = best_shift_corr_torch(ecg_real, pulse_real[:, ch:ch + 1])
        cf = best_shift_corr_torch(ecg_fake, pulse_fake[:, ch:ch + 1])
        loss = loss + (cr - cf).abs().mean()
    return loss / 2.0

print("coupling utilities ready.")

In [ ]:
# ===== 18.5 pulse 생성기 손실 =====
def pulse_gen_loss(pulse_hat, pulse_real, ecg):
    wav = F.l1_loss(pulse_hat, pulse_real)
    deriv = F.l1_loss(pulse_hat[..., 1:] - pulse_hat[..., :-1],
                      pulse_real[..., 1:] - pulse_real[..., :-1])
    Ff = torch.log1p(torch.fft.rfft(pulse_hat,  dim=-1).abs())
    Fr = torch.log1p(torch.fft.rfft(pulse_real, dim=-1).abs())
    freq = F.l1_loss(Ff, Fr)
    coup = ecg_pulse_coupling_loss(ecg, pulse_real, ecg, pulse_hat)
    total = wav + 0.5 * deriv + 0.2 * freq + 0.5 * coup
    logs = {"total": float(total.detach()), "wav": float(wav.detach()),
            "deriv": float(deriv.detach()), "freq": float(freq.detach()),
            "coup": float(coup.detach())}
    return total, logs

print("pulse_gen_loss ready.")

In [ ]:
# ===== 18.6 신경망 pulse 생성기 학습 (실데이터 있으면) =====
from torch.utils.data import TensorDataset, DataLoader
pulse_generator = None
PULSE_METHOD = "rule_based"
if USE_NEURAL_PULSE_GENERATOR and REAL4_AVAILABLE:
    try:
        set_seed(42)
        ecg_in   = X_real_complete[:, 0:2, :].contiguous()   # z-normed ECG
        pulse_tg = X_real_complete[:, 2:4, :].contiguous()   # z-normed PPG/ABP
        n = len(ecg_in)
        perm = torch.randperm(n); n_tr = max(1, int(n * 0.8))
        tr_i, va_i = perm[:n_tr], perm[n_tr:]
        tr_ld = DataLoader(TensorDataset(ecg_in[tr_i], pulse_tg[tr_i]),
                           batch_size=PULSE_GEN_BATCH_SIZE, shuffle=True, drop_last=False)
        va_ld = DataLoader(TensorDataset(ecg_in[va_i], pulse_tg[va_i]),
                           batch_size=PULSE_GEN_BATCH_SIZE, shuffle=False)
        gen = ECGToPulseGenerator().to(PULSE_DEVICE)
        opt = torch.optim.AdamW(gen.parameters(), lr=PULSE_GEN_LR, weight_decay=1e-4)
        best_val = float("inf")
        best_path = os.path.join(PPG_ABP_EXT_DIR, "ecg_to_pulse_best.pt")
        for ep in range(1, PULSE_GEN_EPOCHS + 1):
            gen.train(); tr_logs = []
            for eb, pb in tqdm(tr_ld, desc=f"pulse ep{ep:02d}", leave=False):
                eb = eb.to(PULSE_DEVICE); pb = pb.to(PULSE_DEVICE)
                opt.zero_grad(set_to_none=True)
                loss, logs = pulse_gen_loss(gen(eb), pb, eb)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(gen.parameters(), 1.0)
                opt.step(); tr_logs.append(logs)
            gen.eval(); va_tot = []
            with torch.no_grad():
                for eb, pb in va_ld:
                    eb = eb.to(PULSE_DEVICE); pb = pb.to(PULSE_DEVICE)
                    _, l = pulse_gen_loss(gen(eb), pb, eb); va_tot.append(l["total"])
            trm = {k: float(np.mean([d[k] for d in tr_logs])) for k in tr_logs[0]}
            vm = float(np.mean(va_tot)) if len(va_tot) else trm["total"]
            if vm < best_val:
                best_val = vm; torch.save({"state": gen.state_dict()}, best_path)
            if ep == 1 or ep % 5 == 0 or ep == PULSE_GEN_EPOCHS:
                print(f"ep{ep:03d} train={trm['total']:.4f} (wav {trm['wav']:.3f} der {trm['deriv']:.3f} "
                      f"freq {trm['freq']:.3f} coup {trm['coup']:.3f}) | val={vm:.4f}")
        gen.load_state_dict(torch.load(best_path, map_location=PULSE_DEVICE)["state"]); gen.eval()
        pulse_generator = gen; PULSE_METHOD = "neural"
        print("neural pulse generator OK. best val =", round(best_val, 4))
    except Exception as _e:
        print("neural 학습 실패 -> rule-based 사용. reason:", repr(_e))
        pulse_generator = None; PULSE_METHOD = "rule_based"
else:
    print("neural 비활성/실데이터 없음 -> rule-based 사용")

In [ ]:
# ===== 18.7 (개선) 단순 지연정렬 규칙기반 PPG/ABP 합성 유틸 =====
# 신경망 생성기가 너무 복잡/노이즈가 많아, 단순하고 해석가능한 pulse 합성을 기본으로 사용한다.
# (신경망 코드는 위 셀들에 그대로 보존되며 USE_NEURAL_PULSE_GENERATOR=False 로 비활성)
USE_NEURAL_PULSE_GENERATOR = False
FALLBACK_TO_RULE_BASED     = True
import numpy as np

def smooth_1d_np(x, window):
    """이동평균 평활. x:(...,L) 마지막 축을 평활. window 홀수 권장."""
    window = int(max(1, window))
    x = np.asarray(x, np.float32)
    if window <= 1:
        return x
    k = np.ones(window, dtype=np.float32) / window
    pad = window // 2
    flat = x.reshape(-1, x.shape[-1])
    out = np.empty_like(flat)
    for i in range(flat.shape[0]):
        xp = np.pad(flat[i], (pad, pad), mode="edge")
        out[i] = np.convolve(xp, k, mode="valid")[:flat.shape[-1]]
    return out.reshape(x.shape)

def detect_ecg_event_centers(ecg_batch, fs=250, min_distance_sec=0.28,
                             threshold_percentile=85, max_events=40):
    """ECG 박동(event) 중심 검출. ecg_batch:(N,2,L) -> 샘플별 center index 배열 list."""
    ecg = np.asarray(ecg_batch, np.float32)
    N, C, L = ecg.shape
    min_dist = max(1, int(round(min_distance_sec * fs)))     # 0.28s @250Hz ~ 70 samples
    amp   = np.max(np.abs(ecg), axis=1)                                          # (N,L)
    slope = np.max(np.abs(np.diff(ecg, axis=-1, prepend=ecg[..., :1])), axis=1)  # (N,L)
    env = 0.7 * smooth_1d_np(amp, 21) + 0.3 * smooth_1d_np(slope, 21)            # (N,L)
    centers = []
    for i in range(N):
        e = env[i]
        thr = max(np.percentile(e, threshold_percentile), e.mean() + 0.5 * e.std())
        mid = e[1:-1]
        ismax = (mid >= e[:-2]) & (mid >= e[2:]) & (mid >= thr)
        cand = np.where(ismax)[0] + 1
        cand = cand[(cand >= 20) & (cand <= 2450)]           # 경계 근처 제거
        order = cand[np.argsort(-e[cand])] if len(cand) else cand
        kept = []
        for t in order:                                      # 강한 peak 우선, 최소거리 강제
            t = int(t)
            if all(abs(t - k) >= min_dist for k in kept):
                kept.append(t)
            if len(kept) >= max_events:
                break
        kept = sorted(kept)
        if len(kept) == 0:                                   # fallback: 일정 간격 기본 박동
            step = int(round(fs * 0.8))
            kept = list(range(step, L - step, step))
        centers.append(np.array(kept, dtype=np.int64))
    return centers

def make_ppg_template(fs=250, duration_sec=0.8):
    """매끄럽고 둥근 pulse: 완만한 상승 + 둥근 정점 + 느린 하강 + 약한 dicrotic notch. max=1."""
    L = int(round(duration_sec * fs))
    t = np.linspace(0.0, 1.0, L, dtype=np.float32)
    rise  = 1.0 - np.exp(-t / 0.10)                          # 완만한(둥근) 상승
    decay = np.clip(np.exp(-(t - 0.18) / 0.45), 0.0, 1.0)    # 느린 하강
    base = rise * decay
    dicrotic = 0.12 * np.exp(-((t - 0.55) ** 2) / (2 * 0.05 ** 2))   # 약한 2차 융기
    w = smooth_1d_np((base + dicrotic)[None, :], 9)[0]
    w = w - w.min()
    return (w / (w.max() + 1e-8)).astype(np.float32)

def make_abp_template(fs=250, duration_sec=0.7):
    """PPG 보다 가파른 수축기 상승 + 높은 정점 + 하강 + 작은 dicrotic notch. max=1."""
    L = int(round(duration_sec * fs))
    t = np.linspace(0.0, 1.0, L, dtype=np.float32)
    rise  = 1.0 - np.exp(-t / 0.045)                         # 가파른 상승
    decay = np.clip(np.exp(-(t - 0.10) / 0.40), 0.0, 1.0)
    base = rise * decay
    dicrotic = 0.18 * np.exp(-((t - 0.42) ** 2) / (2 * 0.04 ** 2))   # dicrotic notch/bump
    w = smooth_1d_np((base + dicrotic)[None, :], 5)[0]
    w = w - w.min()
    return (w / (w.max() + 1e-8)).astype(np.float32)

def _place_pulse(track, center, template, amp):
    """template 정점(argmax)을 center 에 정렬해 가산. 경계는 안전하게 잘림."""
    L = track.shape[0]; tl = template.shape[0]
    start = int(center) - int(np.argmax(template))
    s = max(0, start); e = min(L, start + tl)
    if e > s:
        ts = s - start
        track[s:e] += amp * template[ts:ts + (e - s)]

def generate_simple_ppg_abp_from_ecg(ecg_batch, fs=250, delay_range_samples=(25, 75),
                                     ppg_noise_std=0.015, abp_noise_std=0.012, seed=42):
    """ECG event 에 지연정렬된 단순 PPG/ABP 합성. 반환 (N,2,L): [:,0]=PPG, [:,1]=ABP."""
    rng = np.random.default_rng(seed)
    ecg = np.asarray(ecg_batch, np.float32)
    N, C, L = ecg.shape
    ppg_tmpl = make_ppg_template(fs); abp_tmpl = make_abp_template(fs)
    centers = detect_ecg_event_centers(ecg, fs=fs)
    dlo, dhi = delay_range_samples
    tt = np.arange(L, dtype=np.float32) / fs
    out = np.zeros((N, 2, L), dtype=np.float32)
    for i in range(N):
        ppg = np.zeros(L, np.float32); abp = np.zeros(L, np.float32)
        for c in centers[i]:
            delay = int(rng.integers(dlo, dhi + 1))          # 100-300ms 지연 (25-75 samples)
            a_ppg = float(rng.uniform(0.8, 1.3))             # event별 진폭 변동
            a_abp = float(rng.uniform(0.7, 1.4))
            if rng.uniform() < 0.15:                          # 가끔 낮은 ABP (압력 불안정)
                a_abp *= float(rng.uniform(0.5, 0.8))
            _place_pulse(ppg, int(c) + delay, ppg_tmpl, a_ppg)
            _place_pulse(abp, int(c) + delay + int(rng.integers(-5, 6)), abp_tmpl, a_abp)
        # 부드러운 baseline wander (저주파) — random-walk 아님
        f1 = float(rng.uniform(0.05, 0.2)); ph = float(rng.uniform(0, 2 * np.pi))
        ppg += 0.05 * np.sin(2 * np.pi * f1 * tt + ph)
        abp += 0.04 * np.sin(2 * np.pi * f1 * tt + ph + 0.7)
        # 약한 가우시안 노이즈
        ppg += rng.normal(0, ppg_noise_std, L).astype(np.float32)
        abp += rng.normal(0, abp_noise_std, L).astype(np.float32)
        out[i, 0] = ppg; out[i, 1] = abp
    return out

def postprocess_pulse_channels(pulse, fs=250):
    """PPG 강한 평활 / ABP 적당한 평활 + 고주파 억제 + per-window z-norm(안전)."""
    pulse = np.asarray(pulse, np.float32).copy()
    pulse[:, 0] = smooth_1d_np(pulse[:, 0], 27)              # PPG 강하게
    pulse[:, 1] = smooth_1d_np(pulse[:, 1], 11)              # ABP 적당히
    mu = pulse.mean(axis=-1, keepdims=True)
    sd = pulse.std(axis=-1, keepdims=True)
    sd = np.where(sd < 1e-3, 1.0, sd)                        # 미세 std 증폭 방지
    return ((pulse - mu) / sd).astype(np.float32)

def estimate_best_delay_samples(ecg, pulse, fs=250, min_lag=25, max_lag=75):
    """ECG envelope vs pulse envelope 상호상관으로 best lag(samples)·corr 반환. ecg:(2,L) pulse:(L,)."""
    e_env = smooth_1d_np(np.max(np.abs(np.asarray(ecg, np.float32)), axis=0)[None, :], 21)[0]
    p = np.asarray(pulse, np.float32)
    p_env = smooth_1d_np(np.abs(p - p.mean())[None, :], 21)[0]
    e_env = (e_env - e_env.mean()) / (e_env.std() + 1e-8)
    p_env = (p_env - p_env.mean()) / (p_env.std() + 1e-8)
    L = e_env.shape[0]; best_lag, best_c = min_lag, -1e9
    for lag in range(min_lag, max_lag + 1):
        a = e_env[:L - lag]; b = p_env[lag:]
        c = float(np.mean(a * b)) if len(a) else -1e9
        if c > best_c:
            best_c, best_lag = c, lag
    return best_lag, best_c

def _hf_ratio(x, fs=250, cut_hz=8.0):
    """cut_hz 이상 주파수 파워 비율."""
    x = np.asarray(x, np.float32) - float(np.mean(x))
    X = np.fft.rfft(x); p = X.real ** 2 + X.imag ** 2
    f = np.fft.rfftfreq(len(x), d=1.0 / fs)
    return float(p[f >= cut_hz].sum() / (p.sum() + 1e-12))

print("simple delay-aligned PPG/ABP utilities ready.")


In [ ]:
# ===== 18.8 (개선) 단순 지연정렬 PPG/ABP 생성 + classifier 호환 파일 저장 =====
import shutil

# --- (3) 생성된 dangerous ECG 견고하게 로드 ---
if "gen_danger" in globals() and gen_danger is not None:
    ecg_gen = np.asarray(gen_danger, np.float32); print("using in-memory gen_danger")
else:
    ecg_gen = None
    for _p in ["/content/risk_ecg_cvae_outputs/generated/generated_dangerous.npy",
               "/content/drive/MyDrive/vtac_project/outputs/cvae_v4/generated_dangerous.npy"]:
        if os.path.exists(_p):
            ecg_gen = np.load(_p).astype(np.float32); print("loaded", _p); break
    if ecg_gen is None:
        raise FileNotFoundError(
            "Generated ECG dangerous samples were not found. Run the ECG generation cell first.")
# (N,2500,2) -> (N,2,2500) 전치
if ecg_gen.ndim == 3 and ecg_gen.shape[1] != 2 and ecg_gen.shape[2] == 2:
    ecg_gen = np.transpose(ecg_gen, (0, 2, 1))
assert ecg_gen.ndim == 3 and ecg_gen.shape[1] == 2, f"gen ECG must be (N,2,L), got {ecg_gen.shape}"
Ngen = ecg_gen.shape[0]
print("generated ECG:", ecg_gen.shape)

# --- (6-8) 단순 지연정렬 PPG/ABP 생성 + 후처리 ---
pulse_syn = generate_simple_ppg_abp_from_ecg(
    ecg_gen, fs=FS, delay_range_samples=(25, 75),
    ppg_noise_std=0.015, abp_noise_std=0.012, seed=42)
pulse_syn = postprocess_pulse_channels(pulse_syn, fs=FS)
assert pulse_syn.shape == (Ngen, 2, ecg_gen.shape[-1]), f"pulse shape {pulse_syn.shape}"
PULSE_METHOD = "rule_based_simple_pulse"
print("PPG/ABP via SIMPLE rule-based delay-aligned generator")

# --- (11) 4채널 조립 + per-window 정규화 ---
X4_syn = np.concatenate([ecg_gen, pulse_syn], axis=1).astype(np.float32)          # (N,4,L)
X4_syn = (X4_syn - X4_syn.mean(axis=-1, keepdims=True)) / (X4_syn.std(axis=-1, keepdims=True) + 1e-6)

X_syn = torch.tensor(X4_syn, dtype=torch.float32)
y_syn = torch.ones(Ngen, dtype=torch.long)
m_syn = torch.ones(Ngen, 4, dtype=torch.float32)

# --- 검증 assertion ---
assert X4_syn.shape[1] == 4 and X4_syn.shape[-1] == 2500
assert torch.all(m_syn == 1)
assert torch.std(X_syn[:, 2, :]) > 0.01, "PPG channel too flat"
assert torch.std(X_syn[:, 3, :]) > 0.01, "ABP channel too flat"

# --- 저장 (기존 파일 백업) ---
out_path = os.path.join(PPG_ABP_OUTPUT_DIR, "generated_true_v4.pt")
if os.path.exists(out_path):
    bk = os.path.join(PPG_ABP_OUTPUT_DIR, "generated_true_v4_backup_before_simple_pulse.pt")
    if not os.path.exists(bk):
        shutil.copy(out_path, bk); print("기존 파일 백업:", bk)
torch.save({
    "X_syn": torch.tensor(X4_syn, dtype=torch.float32),
    "y_syn": y_syn,
    "m_syn": m_syn,
    "channels": ["ECG1", "ECG2", "PPG", "ABP"],
    "method": "ecg_cvae_plus_simple_delay_aligned_ppg_abp",
    "pulse_generator": "rule_based_simple_pulse",
    "fs": 250,
    "expected_delay_samples": [25, 75],
    "expected_delay_ms": [100, 300],
}, out_path)
print("saved:", out_path, "| X_syn", tuple(X_syn.shape))


In [ ]:
# ===== 18.9 (개선) 단순 pulse 4채널 시각화 =====
import matplotlib.pyplot as plt
SIMPLE_PULSE_PLOTS_DIR = os.path.join(PPG_ABP_OUTPUT_DIR, "ppg_abp_simple_pulse_plots")
os.makedirs(SIMPLE_PULSE_PLOTS_DIR, exist_ok=True)

def plot_4ch_waveform_simple(x4, title=None, save_path=None):
    """x4:(4,L) [ECG1,ECG2,PPG,ABP] 를 세로로 4개 패널로 플롯."""
    x4 = np.asarray(x4, np.float32)
    names  = ["ECG1", "ECG2", "PPG", "ABP"]
    colors = ["tab:blue", "tab:red", "tab:green", "tab:purple"]
    fig, axes = plt.subplots(4, 1, figsize=(13, 8), sharex=True)
    for ax, ch, nm, col in zip(axes, x4, names, colors):
        ax.plot(ch, color=col, lw=0.9); ax.set_ylabel(nm); ax.grid(True, alpha=0.3)
    axes[0].set_title(title or "generated 4-channel (ECG1 / ECG2 / PPG / ABP)")
    axes[-1].set_xlabel("time step")
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=100, bbox_inches="tight")
    plt.show(); plt.close(fig)

_nplot = min(10, int(X_syn.shape[0]))
for i in range(_nplot):
    plot_4ch_waveform_simple(
        X_syn[i].numpy(),
        title=f"generated true #{i} — simple delay-aligned PPG/ABP",
        save_path=os.path.join(SIMPLE_PULSE_PLOTS_DIR, f"sample_{i:02d}.png"))
print(f"saved {_nplot} plots to", SIMPLE_PULSE_PLOTS_DIR)


In [ ]:
# ===== 18.10 (개선) 지연 일관성 + 단순성 지표 + 최종 요약 =====
# --- (9) ECG -> PPG/ABP 지연 일관성 ---
_K = min(200, int(X_syn.shape[0]))
ppg_lags, ppg_cors, abp_lags, abp_cors = [], [], [], []
for i in range(_K):
    _e = X_syn[i, 0:2, :].numpy()
    lp, cp = estimate_best_delay_samples(_e, X_syn[i, 2, :].numpy(), fs=FS)
    la, ca = estimate_best_delay_samples(_e, X_syn[i, 3, :].numpy(), fs=FS)
    ppg_lags.append(lp); ppg_cors.append(cp); abp_lags.append(la); abp_cors.append(ca)
mean_ppg_lag = float(np.mean(ppg_lags)); mean_abp_lag = float(np.mean(abp_lags))
mean_ppg_cor = float(np.mean(ppg_cors)); mean_abp_cor = float(np.mean(abp_cors))
print("=== ECG -> PPG/ABP delay consistency ===")
print(f"mean ECG-PPG delay : {mean_ppg_lag:.1f} samples ({mean_ppg_lag / FS * 1000:.0f} ms)")
print(f"mean ECG-ABP delay : {mean_abp_lag:.1f} samples ({mean_abp_lag / FS * 1000:.0f} ms)")
print(f"mean ECG-PPG corr  : {mean_ppg_cor:.3f}")
print(f"mean ECG-ABP corr  : {mean_abp_cor:.3f}")
print("expected: 25-75 samples (100-300 ms)")

# --- (10) 파형 단순성 지표 ---
def _slope_energy(x): d = np.diff(x); return float(np.mean(d * d))
def _d2_energy(x):    d = np.diff(x, n=2); return float(np.mean(d * d))
def _count_peaks(x, fs=250, dist_sec=0.28):
    md = max(1, int(dist_sec * fs)); m = x[1:-1]
    pk = np.where((m >= x[:-2]) & (m >= x[2:]))[0] + 1
    pk = pk[x[pk] > x.mean() + 0.5 * x.std()]
    kept = []
    for t in sorted(pk, key=lambda t: -x[t]):
        if all(abs(int(t) - k) >= md for k in kept): kept.append(int(t))
    return len(kept)
def _ch_metrics(ch):
    return dict(amp_range=float(ch.max() - ch.min()), std=float(ch.std()),
                slope_e=_slope_energy(ch), d2_e=_d2_energy(ch),
                hf=_hf_ratio(ch, FS, 8.0), npeaks=float(_count_peaks(ch, FS)))
_mi = min(64, int(X_syn.shape[0]))
chmap = {"ECG1": 0, "PPG": 2, "ABP": 3}
agg = {}
for nm, ci in chmap.items():
    mets = [_ch_metrics(X_syn[i, ci, :].numpy()) for i in range(_mi)]
    agg[nm] = {k: float(np.mean([m[k] for m in mets])) for k in mets[0]}
print("\n=== waveform simplicity metrics (mean over %d) ===" % _mi)
print(f"{'metric':12s} {'ECG1':>10s} {'PPG':>10s} {'ABP':>10s}")
for k in ["amp_range", "std", "slope_e", "d2_e", "hf", "npeaks"]:
    print(f"{k:12s} {agg['ECG1'][k]:10.4f} {agg['PPG'][k]:10.4f} {agg['ABP'][k]:10.4f}")
print("기대: PPG/ABP hf < ECG hf | PPG d2_e < ABP d2_e | PPG/ABP not flat & not all-noise")

# --- 최종 assertion ---
assert X_syn.shape[1] == 4 and X_syn.shape[-1] == 2500
assert torch.all(m_syn == 1)
assert torch.std(X_syn[:, 2, :]) > 0.01 and torch.std(X_syn[:, 3, :]) > 0.01

# --- (13) 최종 요약 ---
_final_path = os.path.join(PPG_ABP_OUTPUT_DIR, "generated_true_v4.pt")
print("\n" + "=" * 62)
print("Generated 4-channel synthetic true alarm data with simple delay-aligned PPG/ABP.\n")
print("X_syn shape:", tuple(X_syn.shape))
print("y_syn shape:", tuple(y_syn.shape))
print("m_syn shape:", tuple(m_syn.shape))
print("PPG/ABP method: rule_based_simple_pulse")
print(f"Mean ECG-PPG delay: {mean_ppg_lag:.1f} samples ({mean_ppg_lag / FS * 1000:.0f} ms)")
print(f"Mean ECG-ABP delay: {mean_abp_lag:.1f} samples ({mean_abp_lag / FS * 1000:.0f} ms)")
print(f"Mean ECG-PPG corr: {mean_ppg_cor:.3f}")
print(f"Mean ECG-ABP corr: {mean_abp_cor:.3f}")
print("Saved file:")
print(_final_path)
print("\nNext:")
print("Open Classifier.ipynb and run from the generated data loading section.")
print("=" * 62)
print("\nUpdated PPG/ABP generation to simple ECG-delay-aligned pulse synthesis.")
print("The classifier-compatible file was saved to:")
print("/content/drive/MyDrive/vtac_project/outputs/cvae_v4/generated_true_v4.pt")


## Next step

Now open `Classifier.ipynb` and run from the data loading section onward.

The classifier will load:

`/content/drive/MyDrive/vtac_project/outputs/cvae_v4/generated_true_v4.pt`

This file now contains full 4-channel synthetic true alarm samples:

`ECG1, ECG2, PPG, ABP`

with `m_syn = [1, 1, 1, 1]`.